In [ ]:
import os
os.environ['OPENAI_API_KEY'] = openai_api_key

In [ ]:
%%writefile ast_validator.py
"""ast_validator.py — AST-based structural validation and auto-repair for generated code."""
import ast
import re
from dataclasses import dataclass, field
from typing import Dict, List, Tuple


@dataclass
class ASTIssue:
    line:    int
    kind:    str
    detail:  str
    fixable: bool


@dataclass
class ASTReport:
    file:   str
    issues: List[ASTIssue] = field(default_factory=list)
    score:  int = 10

    @property
    def passed(self) -> bool:
        return self.score >= 6 and not any(i.kind == "syntax_error" for i in self.issues)

    def to_prompt_str(self) -> str:
        if not self.issues:
            return f"AST Validation: PASS (score {self.score}/10)"
        lines = [f"AST Validation: {'PASS' if self.passed else 'FAIL'} (score {self.score}/10)"]
        for issue in self.issues:
            tag = "[AUTO-FIXABLE]" if issue.fixable else "[NEEDS FIX]"
            lines.append(f"  L{issue.line:4d} | {issue.kind:<28} | {tag} {issue.detail}")
        return "\n".join(lines)


class ASTValidator:
    CORRECT_IMPORTS = {
        "HumanMessage":  "langchain_core.messages",
        "AIMessage":     "langchain_core.messages",
        "SystemMessage": "langchain_core.messages",
        "BaseMessage":   "langchain_core.messages",
        "tool":          "langchain_core.tools",
        "ChatOpenAI":    "langchain_openai",
        "StateGraph":    "langgraph.graph",
        "END":           "langgraph.graph",
        "START":         "langgraph.graph",
        "ToolNode":      "langgraph.prebuilt",
        "MemorySaver":   "langgraph.checkpoint.memory",
    }
    BANNED_MODULES = [
        "langchain.schema", "langchain_core.schema",
        "langchain.agents", "langchain.chat_models", "langchain.llms",
    ]
    BANNED_EXACT = ["langchain"]

    def validate(self, source: str, filepath: str = "<generated>") -> ASTReport:
        report = ASTReport(file=filepath)
        try:
            tree = ast.parse(source)
        except SyntaxError as e:
            report.issues.append(ASTIssue(e.lineno or 0, "syntax_error", str(e), False))
            report.score = 0
            return report
        self._check_tool_docstrings(tree, report)
        self._check_imports(tree, report)
        self._check_langgraph_wiring(tree, report, source)
        return report

    def _check_tool_docstrings(self, tree, report):
        for node in ast.walk(tree):
            if not isinstance(node, ast.FunctionDef):
                continue
            decs = [getattr(d, 'id', getattr(d, 'attr', '')) for d in node.decorator_list]
            if 'tool' not in decs:
                continue
            has_doc = (
                node.body
                and isinstance(node.body[0], ast.Expr)
                and isinstance(node.body[0].value, ast.Constant)
                and isinstance(node.body[0].value.value, str)
            )
            if not has_doc:
                report.issues.append(ASTIssue(
                    node.lineno, "missing_tool_docstring",
                    f"@tool '{node.name}' has no docstring — ValueError at import", True))
                report.score -= 2

    def _check_imports(self, tree, report):
        imported_from: Dict[str, str] = {}
        for node in ast.walk(tree):
            if not isinstance(node, ast.ImportFrom):
                continue
            module = node.module or ""
            for alias in node.names:
                imported_from[alias.name] = module
            is_safe = (
                (module.startswith("langchain_core") and module != "langchain_core.schema")
                or module.startswith("langchain_openai")
                or module.startswith("langgraph")
            )
            prefix_banned = any(module == b or module.startswith(b + ".") for b in self.BANNED_MODULES)
            exact_banned  = module in self.BANNED_EXACT
            if (prefix_banned or exact_banned) and not is_safe:
                report.issues.append(ASTIssue(
                    node.lineno, "banned_import",
                    f"banned module '{module}' — use langchain_core.*", True))
                report.score -= 3
        for sym, correct in self.CORRECT_IMPORTS.items():
            if sym in imported_from and imported_from[sym] != correct:
                report.issues.append(ASTIssue(
                    0, "wrong_import_path",
                    f"'{sym}' from '{imported_from[sym]}', should be '{correct}'", True))
                report.score -= 2

    def _check_langgraph_wiring(self, tree, report, source_text: str):
        calls_found = set()
        for node in ast.walk(tree):
            if isinstance(node, ast.Call):
                func = node.func
                name = func.attr if isinstance(func, ast.Attribute) else (
                    func.id if isinstance(func, ast.Name) else "")
                calls_found.add(name)

        for call, msg in [("StateGraph", "StateGraph never constructed"),
                          ("compile",    "workflow.compile() never called")]:
            if call not in calls_found:
                report.issues.append(ASTIssue(0, f"missing_{call.lower()}", msg, False))
                report.score -= 3

        if "ChatOpenAI" not in calls_found:
            report.issues.append(ASTIssue(
                0, "missing_llm",
                "ChatOpenAI never instantiated — agent makes no LLM calls", False))
            report.score -= 4

        if "invoke" not in calls_found and ".invoke(" not in source_text:
            report.issues.append(ASTIssue(
                0, "missing_llm_invoke",
                "llm.invoke() never called — nodes produce no LLM output", False))
            report.score -= 4

        if "from tools import" not in source_text and "import tools" not in source_text:
            report.issues.append(ASTIssue(
                0, "missing_tools_import",
                "tools.py not imported — add: from tools import ..., ALL_TOOLS", False))
            report.score -= 2

        self._check_node_names(tree, report, source_text)

    def _check_node_names(self, tree, report, source_text: str):
        registered: Dict[int, str] = {}
        referenced: Dict[int, str] = {}
        for node in ast.walk(tree):
            if not isinstance(node, ast.Call):
                continue
            method = getattr(node.func, 'attr', '') if isinstance(node.func, ast.Attribute) else ''
            if method == 'add_node' and node.args:
                first = node.args[0]
                if isinstance(first, ast.Constant) and isinstance(first.value, str):
                    registered[node.lineno] = first.value
            elif method in ('add_edge', 'add_conditional_edges') and node.args:
                for arg in node.args:
                    if isinstance(arg, ast.Constant) and isinstance(arg.value, str):
                        val = arg.value
                        if val not in ('__start__', '__end__', 'START', 'END'):
                            referenced[node.lineno] = val

        reg_names = set(registered.values()) | {'__start__', '__end__', 'START', 'END'}
        for lineno, name in referenced.items():
            if name not in reg_names:
                report.issues.append(ASTIssue(
                    lineno, "unknown_node_ref",
                    f"Edge refs '{name}' but add_node('{name}') never called. "
                    f"Registered: {sorted(reg_names - {'__start__','__end__','START','END'})}",
                    True))
                report.score -= 3

        has_start = bool(re.search(r'\.(add_edge|add_conditional_edges)\s*\(\s*START\b', source_text))
        if not has_start and "'__start__'" not in source_text:
            first = sorted(registered.items())[0][1] if registered else "first_node"
            report.issues.append(ASTIssue(
                0, "missing_start_edge",
                f"No edge from START. Add: workflow.add_edge(START, '{first}')", True))
            report.score -= 4

    def auto_repair(self, source: str, report: ASTReport) -> Tuple[str, List[str]]:
        """Apply fixes for auto-fixable issues. Returns (patched_source, fixes_applied)."""
        patches = []
        lines   = source.splitlines(keepends=True)

        for issue in report.issues:
            if issue.kind == "missing_tool_docstring":
                fn_name = re.search(r"@tool '(\w+)'", issue.detail)
                fn_name = fn_name.group(1) if fn_name else None
                if fn_name:
                    for i, line in enumerate(lines):
                        if line.lstrip().startswith("def ") and f"def {fn_name}(" in line:
                            indent = " " * (len(line) - len(line.lstrip()) + 4)
                            lines.insert(i + 1, f'{indent}"""Execute {fn_name} and return result."""\n')
                            patches.append(f"Injected docstring for {fn_name}")
                            break

            elif issue.kind == "banned_import":
                for i, line in enumerate(lines):
                    m = re.match(r"\s*(?:from|import)\s+([\w\.]+)", line.lstrip())
                    if m:
                        mod    = m.group(1)
                        is_bad = mod in self.BANNED_EXACT or any(
                            mod == b or mod.startswith(b + ".") for b in self.BANNED_MODULES)
                        is_safe = (
                            (mod.startswith("langchain_core") and mod != "langchain_core.schema")
                            or mod.startswith("langchain_openai")
                            or mod.startswith("langgraph")
                        )
                        if is_bad and not is_safe:
                            lines[i] = f"# [AUTO-FIXED] {line.rstrip()}\n"
                            patches.append(f"Removed banned import at line {i+1}")

        return "".join(lines), patches

Writing ast_validator.py


In [ ]:
%%writefile code_fixers.py
"""code_fixers.py — Post-processing repairs applied to generated agent code before writing to disk."""
import ast
import hashlib
import re
import sys
import textwrap
from typing import Dict, Tuple

# ── _sanitize_messages source — injected into every generated agent.py ───────
_SANITIZE_SOURCE = '''
def _sanitize_messages(messages: list) -> list:
    """Remove unresolved tool_calls and orphaned ToolMessages to prevent OpenAI 400 errors."""
    from langchain_core.messages import AIMessage, ToolMessage
    covered = {m.tool_call_id for m in messages if isinstance(m, ToolMessage)}
    safe = []
    i = 0
    while i < len(messages):
        msg = messages[i]
        if isinstance(msg, AIMessage) and getattr(msg, "tool_calls", None):
            call_ids = {tc["id"] for tc in msg.tool_calls}
            if call_ids.issubset(covered):
                safe.append(msg)
                i += 1
                collected: set = set()
                while i < len(messages) and isinstance(messages[i], ToolMessage):
                    tm = messages[i]
                    if tm.tool_call_id in call_ids:
                        safe.append(tm)
                        collected.add(tm.tool_call_id)
                    i += 1
                if not call_ids.issubset(collected):
                    while safe and isinstance(safe[-1], ToolMessage):
                        safe.pop()
                    safe[-1] = AIMessage(content=safe[-1].content or "(processing)")
            else:
                safe.append(AIMessage(content=msg.content or "(processing)"))
                i += 1
        elif isinstance(msg, ToolMessage):
            i += 1
        else:
            safe.append(msg)
            i += 1
    return safe
'''

# ── Canonical run.py invocation block ────────────────────────────────────────
_RUN_CANONICAL = textwrap.dedent("""\
    from langchain_core.messages import HumanMessage

    def run(query: str = "Hello, what can you help me with?"):
        config = {"configurable": {"thread_id": "run_1"}}
        initial_state = {"messages": [HumanMessage(content=query)]}
        print(f"Running agent with query: {query!r}\\n")
        seen = []
        for event in app.stream(initial_state, config=config):
            for node_name, update in event.items():
                if isinstance(update, dict) and "messages" in update:
                    for msg in update["messages"]:
                        text = getattr(msg, "content", "")
                        if text and text not in seen:
                            print(f"[{node_name}] {text[:400]}")
                            seen.append(text)
        print("\\nAgent run complete.")
        return seen

    if __name__ == "__main__":
        import sys
        query = " ".join(sys.argv[1:]) or "Hello, what can you help me with?"
        run(query)
""")

_PATH_HEADER = (
    "import sys, os\n"
    "_AGENT_DIR = os.path.abspath(os.path.dirname(__file__))\n"
    "if _AGENT_DIR not in sys.path:\n"
    "    sys.path.insert(0, _AGENT_DIR)\n\n"
)


def fix_run_py(content: str) -> str:
    """
    Fix two classes of LLM defects in generated run.py:
    1. Dotted module imports (output.agent_name.agent) → plain (agent)
    2. Wrong invocation pattern → replace with canonical app.stream() block
    """
    # Strip dotted package imports
    content = re.sub(r"from\s+[\w\.]+\.agent\s+import\s+", "from agent import ", content)
    content = re.sub(r"import\s+[\w\.]+\.agent\b", "import agent", content)

    # Keep only clean import lines before any executable code
    import_lines, seen = [], set()
    for line in content.splitlines():
        s = line.strip()
        if not s or s.startswith("#"):
            continue
        if s.startswith("import ") or s.startswith("from "):
            if "_AGENT_DIR" in s or ("sys.path" in s and "insert" in s):
                continue
            if s not in seen:
                seen.add(s)
                import_lines.append(line)
        else:
            break

    if not any("from agent import" in l or "import agent" in l for l in import_lines):
        import_lines.append("from agent import app")

    return _PATH_HEADER + "\n".join(import_lines) + "\n\n" + _RUN_CANONICAL


def inject_sanitize(source: str) -> str:
    """
    Inject _sanitize_messages() into agent.py.
    - Always injects the function definition after the last top-level import.
    - Rewrites bare `state['messages']` in llm calls to use _sanitize_messages.
    - Safe to call multiple times (checks for existing definition first).
    """
    if "_sanitize_messages" in source:
        # Already present — still rewrite any bare state['messages'] in llm calls
        source = _rewrite_llm_calls(source)
        return source

    lines = source.splitlines(keepends=True)
    last_import = 0
    for i, line in enumerate(lines):
        stripped = line.lstrip()
        if stripped.startswith("import ") or stripped.startswith("from "):
            last_import = i

    # Inject after the last import line
    lines.insert(last_import + 1, _SANITIZE_SOURCE + "\n")
    source = "".join(lines)
    source = _rewrite_llm_calls(source)
    return source


def _rewrite_llm_calls(source: str) -> str:
    """Wrap state['messages'] in _sanitize_messages() in all llm.invoke() patterns."""
    # Pattern: llm*.invoke([...] + state['messages'])
    source = re.sub(
        r"(llm\w*\.invoke\(\[.*?\]\s*\+\s*)state\[(['\"])messages\2\]",
        r"\1_sanitize_messages(state[\2messages\2])",
        source, flags=re.DOTALL
    )
    # Pattern: llm*.invoke(state['messages'])
    source = re.sub(
        r"(llm\w*\.invoke\(\s*)state\[(['\"])messages\2\]",
        r"\1_sanitize_messages(state[\2messages\2])",
        source
    )
    return source


def fix_start_edge(source: str) -> Tuple[str, list]:
    """Ensure graph has workflow.add_edge(START, first_node). Repairs set_entry_point too."""
    repairs = []

    def replace_set_entry(m):
        name = m.group(1).strip().strip("'\"")
        repairs.append(f"set_entry_point → add_edge(START, '{name}')")
        return f"workflow.add_edge(START, '{name}')"

    source = re.sub(r'workflow\.set_entry_point\(([^)]+)\)', replace_set_entry, source)

    has_start = bool(re.search(r'\.(add_edge|add_conditional_edges)\s*\(\s*START\b', source))
    if has_start or "'__start__'" in source:
        return source, repairs

    try:
        tree = ast.parse(source)
    except SyntaxError:
        return source, repairs

    first_node = None
    for node in ast.walk(tree):
        if isinstance(node, ast.Call) and getattr(node.func, 'attr', '') == 'add_node' and node.args:
            arg = node.args[0]
            if isinstance(arg, ast.Constant) and isinstance(arg.value, str):
                name = arg.value
                if 'tool' not in name.lower() and not name.endswith('_tools'):
                    first_node = name
                    break

    if not first_node:
        return source, repairs

    if 'START' not in source:
        source = source.replace(
            'from langgraph.graph import StateGraph, END',
            'from langgraph.graph import StateGraph, END, START'
        )

    inject = f"workflow.add_edge(START, '{first_node}')  # auto-injected\n"
    source = re.sub(r'(app\s*=\s*workflow\.compile)', inject + r'\1', source, count=1)
    repairs.append(f"Injected add_edge(START, '{first_node}')")
    return source, repairs


def fix_node_names(source: str) -> Tuple[str, list]:
    """Repair mismatched node names between add_node() and add_edge() calls."""
    repairs = []
    try:
        tree = ast.parse(source)
    except SyntaxError:
        return source, []

    registered: Dict[str, str] = {}
    fn_to_node:  Dict[str, str] = {}
    for node in ast.walk(tree):
        if not isinstance(node, ast.Call):
            continue
        if getattr(node.func, 'attr', '') == 'add_node' and len(node.args) >= 2:
            na, fa = node.args[0], node.args[1]
            if isinstance(na, ast.Constant) and isinstance(na.value, str):
                nn = na.value
                fn = fa.id if isinstance(fa, ast.Name) else ast.unparse(fa)
                registered[nn] = fn
                fn_to_node[fn] = nn

    if not registered:
        return source, []

    reg_set = set(registered) | {'__start__', '__end__', 'START', 'END'}

    def best_match(bad: str) -> str | None:
        if bad in fn_to_node:                return fn_to_node[bad]
        if bad.endswith('_node'):
            c = bad[:-5]
            if c in reg_set:                 return c
        if bad + '_node' in reg_set:         return bad + '_node'
        starts = [r for r in reg_set if r.startswith(bad) and r != bad]
        if len(starts) == 1:                 return starts[0]
        contained = [r for r in reg_set if bad in r and r != bad]
        if len(contained) == 1:              return contained[0]
        return None

    lines = source.splitlines()
    fixed = []
    for line in lines:
        if not any(m in line for m in ('add_edge', 'add_conditional_edges')):
            fixed.append(line)
            continue

        def replace_name(m: re.Match) -> str:
            q, name = m.group(1), m.group(2)
            if name in reg_set:
                return m.group(0)
            fix = best_match(name)
            if fix:
                repairs.append(f"'{name}' → '{fix}'")
                return f"{q}{fix}{q}"
            return m.group(0)

        fixed.append(re.sub(r'(["\'])([^"\']+)\1', replace_name, line))

    return '\n'.join(fixed), repairs


def compute_fingerprint(code_files: Dict[str, str]) -> str:
    """SHA-256 fingerprint of all file contents + Python + package versions."""
    h = hashlib.sha256()
    for path in sorted(code_files):
        h.update(str(path).encode())
        h.update(str(code_files[path]).encode())   # coerce — value may be dict if LLM wrapped output
    h.update(sys.version.encode())
    for pkg in ["langgraph", "langchain-core", "langchain-openai", "openai"]:
        try:
            import importlib.metadata
            h.update(f"{pkg}=={importlib.metadata.version(pkg)}".encode())
        except Exception:
            h.update(f"{pkg}==unknown".encode())
    fp = h.hexdigest()[:16].upper()
    return f"AF3-{fp[:4]}-{fp[4:8]}-{fp[8:12]}-{fp[12:]}"

Writing code_fixers.py


In [ ]:
%%writefile config.py
"""config.py — Constants, LLM clients, and shared utilities."""
import os
import time

from langchain_openai import ChatOpenAI

# ── Limits ───────────────────────────────────────────────────────────────────
MAX_ITERATIONS    = 25
MAX_FIX_ATTEMPTS  = 3
CRITIC_PASS_SCORE = 6       # out of 10
OUTPUT_ROOT       = "./output"
_SLIM_CONTEXT_ROLES = {"Tester", "Constitutional Critic", "Fixer"}

# ── LLM ──────────────────────────────────────────────────────────────────────
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "YOUR_API_KEY_HERE")
llm = ChatOpenAI(openai_api_key=OPENAI_API_KEY, model="gpt-4o", temperature=0)


def invoke_with_backoff(llm_bound, messages, max_retries: int = 6):
    """Exponential backoff on RateLimitError (2, 4, 8 … 64 s)."""
    import openai
    for attempt in range(max_retries):
        try:
            return llm_bound.invoke(messages)
        except openai.RateLimitError:
            if attempt == max_retries - 1:
                raise
            wait = 2 ** (attempt + 1)
            print(f"  [RateLimit] sleeping {wait}s (retry {attempt+1}/{max_retries-1})…")
            time.sleep(wait)
        except Exception:
            raise

Writing config.py


In [ ]:
%%writefile graph.py
"""graph.py — Assembles the AgentForge LangGraph StateGraph."""
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.prebuilt import ToolNode

from config import MAX_ITERATIONS
from nodes import (
    analyze_logs,
    create_agent_node,
    execute_script,
    generate_mermaid_diagram,
    install_requirements,
    list_output_files,
    read_file,
    run_ast_validation,
    run_syntax_check,
    search_documentation,
    tool_generator_node,
    tool_planner_node,
    write_code_files_node,
)
from prompts import (
    MULTI_ARCH_PROMPT,
    ORCHESTRATOR_PROMPT,
    SINGLE_ARCH_PROMPT,
    TESTER_PROMPT,
    build_developer_prompt,
)
from state import AgentState


def has_tool_calls(state: AgentState) -> bool:
    last = state["messages"][-1]
    return bool(hasattr(last, "tool_calls") and last.tool_calls)


def build_graph():
    workflow = StateGraph(AgentState)

    # ── Agent nodes ──────────────────────────────────────────────────────────
    workflow.add_node("orchestrator",    create_agent_node("Orchestrator",            [],                                               ORCHESTRATOR_PROMPT))
    workflow.add_node("single_arch",     create_agent_node("Single-Agent Architect",  [search_documentation, generate_mermaid_diagram], SINGLE_ARCH_PROMPT))
    workflow.add_node("multi_arch",      create_agent_node("Multi-Agent Architect",   [search_documentation, generate_mermaid_diagram], MULTI_ARCH_PROMPT))

    # tool_planner is kept in imports/nodes for reference but removed from the active graph flow
    workflow.add_node("tool_generator",  tool_generator_node)
    workflow.add_node("developer",       create_agent_node("Developer",               [search_documentation],                          build_developer_prompt))
    workflow.add_node("write_files",     write_code_files_node)
    workflow.add_node("tester",          create_agent_node("Tester",
        [run_syntax_check, install_requirements, run_ast_validation,
         execute_script, analyze_logs, read_file, list_output_files],
        TESTER_PROMPT))

    # ── Tool nodes ───────────────────────────────────────────────────────────
    workflow.add_node("single_arch_tools", ToolNode([search_documentation, generate_mermaid_diagram]))
    workflow.add_node("multi_arch_tools",  ToolNode([search_documentation, generate_mermaid_diagram]))
    workflow.add_node("dev_tools",         ToolNode([search_documentation]))
    workflow.add_node("test_tools",        ToolNode([
         run_syntax_check, install_requirements, run_ast_validation,
         execute_script, analyze_logs, read_file, list_output_files,
    ]))

    # ── Edges ────────────────────────────────────────────────────────────────
    workflow.add_edge(START, "orchestrator")

    def route_orch(state):
        return "single_arch" if state.get("architecture_type", "single") == "single" else "multi_arch"

    workflow.add_conditional_edges("orchestrator", route_orch,
                                   {"single_arch": "single_arch", "multi_arch": "multi_arch"})

    # Routing logic: If tools are called, go to ToolNode. If not, go to Generator (Ablation: Skip Planner)
    def route_single(state):
        return "single_arch_tools" if has_tool_calls(state) else "tool_generator"

    def route_multi(state):
        return "multi_arch_tools" if has_tool_calls(state) else "tool_generator"

    workflow.add_conditional_edges(
        "single_arch",
        route_single,
        {
            "single_arch_tools": "single_arch_tools",
            "tool_generator": "tool_generator"
        }
    )

    workflow.add_conditional_edges(
        "multi_arch",
        route_multi,
        {
            "multi_arch_tools": "multi_arch_tools",
            "tool_generator": "tool_generator"
        }
    )

    # Tool return loops
    workflow.add_edge("single_arch_tools", "single_arch")
    workflow.add_edge("multi_arch_tools",  "multi_arch")

    # Tool pipeline: Skip planner, go straight generator -> developer
    workflow.add_edge("tool_generator", "developer")

    def route_dev(state):
        return "dev_tools" if has_tool_calls(state) else "write_files"

    workflow.add_conditional_edges("developer", route_dev,
                                   {"dev_tools": "dev_tools", "write_files": "write_files"})
    workflow.add_edge("dev_tools", "developer")

    workflow.add_edge("write_files", "tester")

    def route_tester(state):
        if has_tool_calls(state):
            return "test_tools"
        report = state.get("test_report", "")
        if state.get("iteration_count", 0) >= MAX_ITERATIONS:  return END
        if "VERDICT: PASS" in report.upper():                   return END
        return END

    workflow.add_conditional_edges("tester", route_tester,
                                   {"test_tools": "test_tools", END: END})
    workflow.add_edge("test_tools", "tester")

    return workflow.compile(checkpointer=MemorySaver())

Overwriting graph.py


In [ ]:
%%writefile main.py
"""main.py — Entry point for AgentForge v3.

Usage in Colab:
    from main import run_agent_forge
    run_agent_forge("A financial research assistant that fetches stock news and writes a brief.")

Or from CLI:
    python main.py "Your task description here"
"""
import os
import sys

from langchain_core.messages import HumanMessage

from config import OUTPUT_ROOT
from graph import build_graph
from state import AgentState


def run_agent_forge(task: str, thread_id: str = "agentforge_v3") -> dict:
    """Run the full AgentForge pipeline. Returns the final state dict."""
    print(f"\n{'═'*65}")
    print("  AgentForge v3 — Real Tool Generation")
    print(f"  Task: {task}")
    print(f"{'═'*65}\n")

    app    = build_graph()
    config = {"configurable": {"thread_id": thread_id}}

    initial_state: AgentState = {
        "messages":           [HumanMessage(content=task)],
        "task":               task,
        "agent_name":         "my_agent",
        "architecture_type":  "",
        "architecture_spec":  "",
        "tool_specs":         "[]",
        "tools_py":           "",
        "tool_names":         "",
        "tool_packages":      "[]",
        "code_files":         "",
        "ast_report":         "",
        "critic_score":       0,
        "critic_feedback":    "",
        "test_report":        "",
        "fix_attempts":       0,
        "iteration_count":    0,
        "output_dir":         OUTPUT_ROOT,
        "fingerprint":        "",
        "final_summary":      "",
    }

    final_state = None
    step = 0
    for event in app.stream(initial_state, config=config, stream_mode="values"):
        step += 1
        final_state = event
        if event.get("messages"):
            last      = event["messages"][-1]
            kind      = type(last).__name__
            preview   = str(last.content or "(tool call)")[:220]
            score_tag = f" [Critic:{event.get('critic_score','?')}/10]" if event.get("critic_score") else ""
            print(f"[Step {step:02d}]{score_tag} [{kind}] {preview}")

    # ── Final summary ─────────────────────────────────────────────────────────
    print(f"\n{'═'*65}")
    print("  AgentForge v3 Complete")
    if final_state:
        out    = final_state.get("output_dir",        OUTPUT_ROOT)
        name   = final_state.get("agent_name",        "?")
        arch   = final_state.get("architecture_type", "?")
        rep    = final_state.get("test_report",       "")
        fp     = final_state.get("fingerprint",       "N/A")
        score  = final_state.get("critic_score",      "N/A")
        tools  = final_state.get("tool_names",        "N/A")
        passed = "VERDICT: PASS" in rep.upper()

        print(f"  Agent        : {name}")
        print(f"  Architecture : {arch}")
        print(f"  Tools        : {tools}")
        print(f"  Critic Score : {score}/10")
        print(f"  Test Verdict : {'✓ PASS' if passed else '✗ FAIL / Incomplete'}")
        print(f"  Fingerprint  : {fp}")
        print(f"\n  Generated files:")
        if os.path.exists(out):
            for root, _, fnames in os.walk(out):
                if "venv" in root:
                    continue
                for f in fnames:
                    print(f"    {os.path.relpath(os.path.join(root, f))}")
    print(f"{'═'*65}\n")
    return final_state or {}


# ── CLI ───────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    _args = sys.argv[1:]
    _is_jupyter = (
        not _args
        or (len(_args) == 2 and _args[0] == "-f")
        or (_args and _args[0].endswith(".json") and "kernel" in _args[0])
    )
    if _args and not _is_jupyter:
        user_task = " ".join(_args)
    else:
        user_task = input("Describe the agent you want to build:\n> ").strip()
        if not user_task:
            user_task = (
                "A financial research assistant that searches for stock news, "
                "fetches current prices, and writes a structured investment brief."
            )
    run_agent_forge(user_task)

Overwriting main.py


In [ ]:
%%writefile nodes.py
"""nodes.py — All LangGraph node functions for the AgentForge pipeline."""
import json
import os
import re
import subprocess
import sys

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool

from ast_validator import ASTValidator
from code_fixers import (
    compute_fingerprint,
)
from config import (
    MAX_FIX_ATTEMPTS,
    MAX_ITERATIONS,
    OPENAI_API_KEY,
    OUTPUT_ROOT,
    _SLIM_CONTEXT_ROLES,
    invoke_with_backoff,
    llm,
)
from prompts import TOOL_PLANNER_PROMPT, build_developer_prompt
from state import AgentState
from tool_gen import build_tools_py, generate_tools_for_task


# ═══════════════════════════════════════════════════════════════════════════════
# AGENTFORGE META-TOOLS  (used by pipeline nodes, not by generated agents)
# ═══════════════════════════════════════════════════════════════════════════════

@tool
def search_documentation(query: str) -> str:
    """Search for LangGraph or LangChain documentation snippets."""
    docs = {
        "langgraph": (
            "LangGraph 2025: StateGraph + TypedDict. workflow.add_node(), "
            "workflow.compile(checkpointer=MemorySaver()). ToolNode for tool routing. "
            "add_conditional_edges for branching."
        ),
        "tool": (
            "Tools: @tool decorator from langchain_core.tools. "
            "EVERY @tool MUST have a docstring. llm.bind_tools(tools)."
        ),
        "state": "State: TypedDict. Annotated[List[BaseMessage], operator.add] for messages.",
    }
    return "\n\n".join(v for k, v in docs.items() if k in query.lower()) \
        or "Use standard LangGraph functional patterns."


@tool
def generate_mermaid_diagram(description: str) -> str:
    """Convert a plain-text flow description into a Mermaid flowchart."""
    lines = [ln.strip() for ln in description.strip().splitlines() if ln.strip()]
    out = ["graph TD"]
    for line in lines:
        parts = re.split(r"\s*-+>\s*", line)
        for i in range(len(parts) - 1):
            out.append(f"    {parts[i].strip().replace(' ','_')} --> {parts[i+1].strip().replace(' ','_')}")
    return "\n".join(out)


@tool
def read_file(filepath: str) -> str:
    """Read and return the contents of a file."""
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        return f"File not found: {filepath}"


@tool
def list_output_files(output_dir: str) -> str:
    """List all files recursively under a directory, excluding venv."""
    if not os.path.exists(output_dir):
        return f"Not found: {output_dir}"
    files = []
    for root, _, fnames in os.walk(output_dir):
        if "venv" in root:
            continue
        for f in fnames:
            files.append(os.path.relpath(os.path.join(root, f), "."))
    return "\n".join(files) or "No files found."


@tool
def install_requirements(requirements_file: str) -> str:
    """Install packages from a requirements.txt into the current Python environment."""
    abs_req = os.path.abspath(requirements_file)
    if not os.path.exists(abs_req):
        return (
            f"SKIP: requirements.txt not found: {abs_req}\n"
            "This usually means write_files failed. Check the previous step for errors."
        )
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", abs_req, "-q"],
        capture_output=True, text=True
    )
    return f"Installed from {abs_req}" if result.returncode == 0 \
        else f"Install failed:\n{result.stderr[:2000]}"


@tool
def execute_script(script_path: str, timeout: int = 60) -> str:
    """Run a Python script with the current interpreter and return stdout/stderr."""
    abs_script = os.path.abspath(script_path)
    if not os.path.exists(abs_script):
        return f"Script not found: {abs_script}"
    try:
        result = subprocess.run(
            [sys.executable, abs_script],
            capture_output=True, text=True, timeout=timeout,
            cwd=os.path.dirname(abs_script),
        )
        return (f"Exit Code: {result.returncode}\n"
                f"STDOUT:\n{result.stdout[:3000]}\n"
                f"STDERR:\n{result.stderr[:2000]}")
    except subprocess.TimeoutExpired:
        return f"Exit Code: 1\nScript timed out after {timeout}s"


@tool
def analyze_logs(logs: str) -> str:
    """Analyze execution logs for structural failures. Auth errors are NOT failures."""
    sanitized = re.sub(
        r"(AuthenticationError|Incorrect API key|openai\.AuthenticationError"
        r"|RateLimitError|APIConnectionError)[^\n]*", "", logs
    )
    issues = []
    for pat, label in [
        (r"ImportError|ModuleNotFoundError", "Missing dependency"),
        (r"SyntaxError",                     "Syntax error"),
        (r"AttributeError",                  "Attribute/API mismatch"),
        (r"TypeError",                       "Type error"),
        (r"NameError",                       "Undefined name"),
        (r"ValueError.*docstring",           "Missing @tool docstring"),
    ]:
        if re.search(pat, sanitized):
            issues.append(label)
    if issues:
        return f"FAIL | {'; '.join(issues)}\n\nLog:\n{logs[:1500]}"
    return "PASS | No critical errors."


@tool
def run_syntax_check(filepath: str) -> str:
    """Run py_compile on a Python file to check for syntax errors."""
    import py_compile
    abs_path = os.path.abspath(filepath)
    if not os.path.exists(abs_path):
        return (
            f"SKIP: file not found: {abs_path}\n"
            "This usually means write_files failed. Check the previous step for errors."
        )
    try:
        py_compile.compile(abs_path, doraise=True)
        return f"Syntax OK: {filepath}"
    except py_compile.PyCompileError as e:
        return f"Syntax error in {filepath}:\n{str(e)}"


@tool
def run_ast_validation(filepath: str) -> str:
    """Deep AST structural validation — LLM calls, imports, wiring, tool docstrings."""
    abs_path = os.path.abspath(filepath)
    if not os.path.exists(abs_path):
        return (
            f"SKIP: file not found: {abs_path}\n"
            "This usually means write_files failed. Check the previous step for errors."
        )
    try:
        with open(abs_path, "r", encoding="utf-8") as f:
            source = f.read()
    except Exception as e:
        return f"Could not read {filepath}: {e}"
    return ASTValidator().validate(source, filepath).to_prompt_str()


# ═══════════════════════════════════════════════════════════════════════════════
# AGENT NODE FACTORY
# ═══════════════════════════════════════════════════════════════════════════════

def create_agent_node(role: str, tools: list, system_prompt_or_factory):
    """Build a LangGraph node function for the given role."""
    llm_bound = llm.bind_tools(tools) if tools else llm

    def node(state: AgentState) -> dict:
        sys_prompt = (
            system_prompt_or_factory(state)
            if callable(system_prompt_or_factory)
            else system_prompt_or_factory
        )

        is_slim   = role in _SLIM_CONTEXT_ROLES
        arch_spec = "(see earlier messages)" if is_slim \
            else state.get("architecture_spec", "Not yet designed")

        ctx = (
            "\n\n### SHARED CONTEXT ###\n"
            f"Task          : {state['task']}\n"
            f"Agent Name    : {state.get('agent_name', 'my_agent')}\n"
            f"Architecture  : {state.get('architecture_type', 'undecided')}\n"
            f"Output Dir    : {state.get('output_dir', OUTPUT_ROOT)}\n"
            f"Fix Attempts  : {state.get('fix_attempts', 0)}/{MAX_FIX_ATTEMPTS}\n"
            f"Iteration     : {state.get('iteration_count', 0)}/{MAX_ITERATIONS}\n"
            f"Critic Score  : {state.get('critic_score', 'N/A')}/10\n"
            f"TOOL_NAMES    : {state.get('tool_names', '')}\n"
            f"TOOL_PACKAGES : {state.get('tool_packages', '[]')}\n"
            f"\nArchitecture Spec:\n{arch_spec}\n"
            f"\nAST Report:\n{state.get('ast_report', 'Not yet validated')}\n"
            f"\nCritic Feedback:\n{state.get('critic_feedback', 'Not yet reviewed')}\n"
            f"\nLast Test Report:\n{state.get('test_report', 'No tests run yet')}"
        )

        history = list(state["messages"])
        if is_slim and len(history) > 6:
            history = history[-6:]
        while history and isinstance(history[0], ToolMessage):
            history = history[1:]

        messages_to_send = [SystemMessage(content=sys_prompt + ctx)] + history
        res = invoke_with_backoff(llm_bound, messages_to_send)

        update: dict = {
            "messages":        [res],
            "iteration_count": state.get("iteration_count", 0) + 1,
        }

        if not (hasattr(res, "tool_calls") and res.tool_calls):
            content = res.content or ""

            if role == "Orchestrator":
                try:
                    raw      = re.search(r"\{[^{}]+\}", content, re.DOTALL).group()
                    decision = json.loads(raw)
                    update["architecture_type"] = decision.get("architecture_type", "single")
                    raw_name = decision.get("agent_name", "my_agent")
                    update["agent_name"]  = re.sub(r"[^a-z0-9_]", "_", raw_name.lower()).strip("_") or "my_agent"
                    update["output_dir"]  = os.path.join(OUTPUT_ROOT, update["agent_name"])
                except Exception:
                    update["architecture_type"] = "single"
                    update["agent_name"]         = "my_agent"
                    update["output_dir"]          = os.path.join(OUTPUT_ROOT, "my_agent")

            elif role in ("Single-Agent Architect", "Multi-Agent Architect"):
                update["architecture_spec"] = content

            elif role in ("Developer", "Fixer"):
                try:
                    cleaned = re.sub(r"```(?:json)?", "", content).replace("```", "").strip()
                    parsed  = json.loads(cleaned)
                    # LLM sometimes wraps output: {"files": {path: content}}
                    # Unwrap any single-key wrapper whose value is a dict of file paths
                    if isinstance(parsed, dict):
                        for wrapper_key in ("files", "output", "code", "result"):
                            inner = parsed.get(wrapper_key)
                            if isinstance(inner, dict) and all(
                                isinstance(k, str) and ("/" in k or k.endswith(".py") or k.endswith(".txt") or k.endswith(".md"))
                                for k in inner
                            ):
                                parsed = inner
                                break
                    update["code_files"] = json.dumps(parsed)
                except Exception:
                    update["code_files"] = content
                if role == "Fixer":
                    update["fix_attempts"] = state.get("fix_attempts", 0) + 1

            elif role == "Constitutional Critic":
                try:
                    cleaned = re.sub(r"```(?:json)?", "", content).replace("```", "").strip()
                    scores  = json.loads(cleaned)
                    update["critic_score"]    = scores.get("total", 0)
                    update["critic_feedback"] = (
                        f"Score: {scores.get('total',0)}/10 | Verdict: {scores.get('verdict','FAIL')}\n"
                        f"Issues: {'; '.join(scores.get('critical_issues',[]))}\n"
                        f"Suggestions: {'; '.join(scores.get('suggestions',[]))}"
                    )
                except Exception:
                    update["critic_score"]    = 5
                    update["critic_feedback"] = content[:300]

            elif role == "Tester":
                update["test_report"] = content
                if "VERDICT: PASS" in content.upper():
                    update["final_summary"] = content

        return update

    return node


# ═══════════════════════════════════════════════════════════════════════════════
# TOOL PLANNER NODE
# ═══════════════════════════════════════════════════════════════════════════════

def tool_planner_node(state: AgentState) -> dict:
    """
    Plan task-specific tools for the generated agent.

    Priority:
    1. Extract TOOLS_JSON from arch spec (architect-embedded list)
    2. Filter out any AgentForge internal tool names that leaked in
    3. If too few tools remain after filtering, re-plan from scratch using the task
    4. Always validate against the task before returning
    """
    arch_spec = state.get("architecture_spec", "")
    task      = state.get("task", "")

    # ── AgentForge internal tools that must NEVER become generated agent tools ─
    _INTERNAL_TOOL_NAMES = {
        "search_documentation", "generate_mermaid_diagram", "read_file", "write_file",
        "list_output_files", "list_files_recursive", "install_requirements",
        "execute_script", "execute_in_venv", "analyze_logs", "run_syntax_check",
        "run_ast_validation", "get_system_metrics", "write_local_file", "read_local_file",
    }

    def _is_internal(name: str) -> bool:
        return name.lower().strip() in _INTERNAL_TOOL_NAMES

    def _llm_plan_tools(reason: str) -> list[dict]:
        """Ask LLM to plan tools from scratch for the task."""
        print(f"  [ToolPlanner] {reason} — asking LLM to plan task-specific tools…")
        prompt = (
            f"Task: {task}\n\n"
            f"Architecture notes:\n{arch_spec[:1000]}\n\n"
            "What tools does this agent need to accomplish the task?\n"
            "Think about the real actions needed: data fetching, computation, analysis.\n"
            "Return ONLY a JSON array (no markdown):\n"
            '[{"name": "snake_case_tool_name", "purpose": "what it does for this task"}, ...]'
        )
        resp = invoke_with_backoff(llm, [
            SystemMessage(content=TOOL_PLANNER_PROMPT),
            HumanMessage(content=prompt),
        ])
        try:
            cleaned = re.sub(r"```(?:json)?", "", resp.content or "").replace("```", "").strip()
            specs   = json.loads(cleaned)
            assert isinstance(specs, list) and len(specs) > 0
            # Filter internal names from LLM output too
            return [s for s in specs if not _is_internal(s.get("name", ""))]
        except Exception:
            return [{"name": "web_search", "purpose": f"search the web for information about {task}"}]

    # ── Step 1: Try TOOLS_JSON from architect ──────────────────────────────────
    specs = []
    match = re.search(r"TOOLS_JSON:\s*(\[.*?\])", arch_spec, re.DOTALL)
    if match:
        try:
            raw_specs = json.loads(match.group(1))
            if isinstance(raw_specs, list) and all("name" in s for s in raw_specs):
                specs = raw_specs
        except Exception:
            pass

    # ── Step 2: Filter internal names ─────────────────────────────────────────
    internal_found = [s["name"] for s in specs if _is_internal(s.get("name", ""))]
    if internal_found:
        print(f"  [ToolPlanner] Filtered internal tools from arch spec: {internal_found}")
    specs = [s for s in specs if not _is_internal(s.get("name", ""))]

    # ── Step 3: Re-plan if too few valid specs remain ─────────────────────────
    if len(specs) == 0:
        specs = _llm_plan_tools("No task-specific tools in arch spec")
    elif len(specs) == 1 and internal_found:
        # Architect mostly gave us internal tools — re-plan properly
        specs = _llm_plan_tools("Arch spec had only internal tools")

    msg = f"Tool Planner: {len(specs)} tools: {[s['name'] for s in specs]}"
    print(f"  [ToolPlanner] {msg}")
    return {"messages": [AIMessage(content=msg)], "tool_specs": json.dumps(specs)}


# ═══════════════════════════════════════════════════════════════════════════════
# TOOL GENERATOR NODE
# ═══════════════════════════════════════════════════════════════════════════════

def tool_generator_node(state: AgentState) -> dict:
    """Generate real @tool implementations for every tool in tool_specs."""
    task      = state.get("task", "")
    specs_raw = state.get("tool_specs", "[]")
    try:
        specs = json.loads(specs_raw)
    except Exception:
        specs = [{"name": "web_search", "purpose": "search the web"}]

    print(f"\n  [ToolGenerator] Generating {len(specs)} tool(s)…")
    for s in specs:
        print(f"    → {s['name']}: {s['purpose']}")

    tool_defs = generate_tools_for_task(task, specs)
    # build_tools_py now returns (source, actual_fn_names)
    tools_py, actual_names = build_tools_py(tool_defs)
    names    = ", ".join(actual_names)
    packages = list({
        pkg for t in tool_defs
        for pkg in t.get("packages", [])
        # Strip stdlib/built-in names that are not pip packages
        if pkg not in ("os", "sys", "re", "json", "io", "pathlib", "subprocess",
                       "stdlib", "collections", "itertools", "functools", "typing",
                       "datetime", "math", "random", "hashlib", "base64", "urllib")
    })

    print(f"  [ToolGenerator] Done. Actual fn names: [{names}]  Extra packages: {packages}")
    msg = f"Tool Generator: {len(tool_defs)} real tools: [{names}]\nExtra packages: {packages}"
    return {
        "messages":      [AIMessage(content=msg)],
        "tools_py":      tools_py,
        "tool_names":    names,        # ← actual Python function names for import
        "tool_packages": json.dumps(packages),
    }


# ═══════════════════════════════════════════════════════════════════════════════
# FILE WRITER NODE
# ═══════════════════════════════════════════════════════════════════════════════


def write_code_files_node(state: AgentState) -> dict:
    """
    Hybrid file writer:
    - agent.py: fixed skeleton + LLM-generated node bodies (via generate_nodes)
                + surgical repair (via repair_agent_py)
    - run.py: fixed template always
    - tools.py: pre-generated real implementations always
    - requirements.txt: fixed template (no stdlib packages)
    - README.md: from LLM spec or fixed template
    """
    from agent_template import (
        assemble_agent_py,
        build_readme,
        build_requirements_txt,
        build_run_py,
        generate_nodes,
        repair_agent_py,
    )

    agent_name     = state.get("agent_name",  "my_agent")
    output_dir     = state.get("output_dir",  os.path.join(OUTPUT_ROOT, agent_name))
    tools_py_src   = state.get("tools_py",    "")
    tool_names_str = state.get("tool_names",  "")
    tool_packages  = json.loads(state.get("tool_packages", "[]"))
    arch_spec      = state.get("architecture_spec", "")
    arch_type      = state.get("architecture_type", "single")
    task           = state.get("task", "")

    tool_names = [n.strip() for n in tool_names_str.split(",") if n.strip()] if tool_names_str else []

    written, errors, repairs_log, ast_summaries = [], [], [], []
    validator   = ASTValidator()
    fingerprint = "ERROR"
    output_dir_fwd = output_dir.replace("\\", "/")

    try:
        # ── Step 1: LLM generates node bodies, prompts, state fields ─────────
        print(f"  [WriteFiles] Generating node bodies for '{agent_name}'...")
        node_spec = generate_nodes(task, tool_names, arch_spec, arch_type)

        # ── Step 2: Assemble agent.py from template + LLM parts ──────────────
        agent_py = assemble_agent_py(tool_names=tool_names, node_spec=node_spec)

        # ── Step 3: Surgical repair on assembled agent.py ────────────────────
        agent_py, reps = repair_agent_py(agent_py)
        repairs_log.extend([f"  repair: {r}" for r in reps])

        # ── Step 4: Build all other files from fixed templates ────────────────
        readme_desc = f"An AI agent for: {task}"
        files = {
            f"{output_dir_fwd}/agent.py":         agent_py,
            f"{output_dir_fwd}/run.py":            build_run_py(agent_name),
            f"{output_dir_fwd}/requirements.txt":  build_requirements_txt(tool_packages),
            f"{output_dir_fwd}/README.md":         build_readme(agent_name, readme_desc, tool_names),
            f"{output_dir_fwd}/tools.py":          tools_py_src if tools_py_src else (
                "# tools.py — No tools for this task\n"
                "from langchain_core.tools import tool\n\n"
                "ALL_TOOLS = []\n"
            ),
        }

        fingerprint = compute_fingerprint(files)
        for k in files:
            files[k] = files[k].replace("{{FINGERPRINT}}", fingerprint)

        # ── Step 5: AST validate (informational + final auto-repair pass) ─────
        for k, content in files.items():
            if not k.endswith(".py"):
                continue
            report = validator.validate(content, k)
            ast_summaries.append(f"[{os.path.basename(k)}]\n{report.to_prompt_str()}")
            if report.issues:
                patched, auto_fixes = validator.auto_repair(content, report)
                if auto_fixes:
                    files[k] = patched
                    repairs_log.extend([f"  AST: {f}" for f in auto_fixes])

        # ── Step 6: Write all files ───────────────────────────────────────────
        for path, content in files.items():
            abs_path = os.path.abspath(path)
            os.makedirs(os.path.dirname(abs_path), exist_ok=True)
            with open(abs_path, "w", encoding="utf-8") as f:
                f.write(content)
            written.append(path)

    except Exception as e:
        import traceback
        errors.append(f"{e}\n{traceback.format_exc()[:800]}")

    parts = []
    if written:
        parts.append(f"Files written ({len(written)}):\n" + "\n".join(f"  {p}" for p in written))
    if repairs_log:
        parts.append("Repairs applied:\n" + "\n".join(repairs_log))
    if ast_summaries:
        parts.append("AST Summary:\n" + "\n\n".join(ast_summaries))
    if errors:
        parts.append(f"Errors:\n" + "\n".join(errors))

    return {
        "messages":    [AIMessage(content="\n\n".join(parts) or "No files written.")],
        "ast_report":  "\n\n".join(ast_summaries) if ast_summaries else "No Python files validated.",
        "fingerprint": fingerprint,
    }

Overwriting nodes.py


In [ ]:
%%writefile prompts.py
"""prompts.py — All system prompts and dynamic prompt builders."""

# ── Orchestrator ──────────────────────────────────────────────────────────────
ORCHESTRATOR_PROMPT = (
    "You are the Orchestrator of AgentForge v3.\n\n"
    "Analyze the user's task and decide:\n"
    "1. SINGLE (one agent loop) vs MULTI (multiple specialized agents)\n"
    "2. A short snake_case name\n\n"
    "SINGLE: simple tasks, one tool, chatbots.\n"
    "MULTI: complex pipelines, multiple roles (search+analyze+write), specialist agents.\n\n"
    "Respond ONLY with JSON:\n"
    '{"architecture_type": "single", "agent_name": "snake_case_name", "reasoning": "one sentence"}'
)

# ── Tool Planner ──────────────────────────────────────────────────────────────
TOOL_PLANNER_PROMPT = (
    "You are a Tool Planner for an AI agent system.\n\n"
    "Your job: decide which tools the agent needs to accomplish the GIVEN TASK.\n\n"
    "Think about what the agent actually needs to DO:\n"
    "- What data does it need to fetch or compute?\n"
    "- What external actions does it need to perform?\n"
    "- What would a human expert use to solve this task?\n\n"
    "Rules:\n"
    "- 2–5 tools maximum — only tools directly necessary for the task\n"
    "- Tools must be implementable with: requests, beautifulsoup4, duckduckgo-search, or Python stdlib\n"
    "- NO tools requiring private API keys\n"
    "- Use DuckDuckGo for any web search or research needs\n"
    "- Tool names must be task-specific snake_case (e.g. analyze_image_url, check_face_landmarks)\n\n"
    "FORBIDDEN tool names (these are AgentForge internal tools, NOT for generated agents):\n"
    "search_documentation, generate_mermaid_diagram, read_file, write_file, list_output_files,\n"
    "install_requirements, execute_script, analyze_logs, run_syntax_check, run_ast_validation\n\n"
    "Respond ONLY with a JSON array:\n"
    '[{"name": "snake_case_name", "purpose": "specific description of what it does for this task"}, ...]'
)

# ── Architect ─────────────────────────────────────────────────────────────────
SINGLE_ARCH_PROMPT = (
    "You are a Single-Agent Architect designing a real AI agent for a specific task.\n\n"
    "Design a production-ready LangGraph ReAct agent for the task given in context.\n\n"
    "Produce:\n"
    "1. Mermaid diagram: START -> agent <-> tools -> END\n"
    "2. State TypedDict fields (always include messages + task-specific output fields)\n"
    "3. System prompt for the agent (specific, task-focused, NOT generic)\n"
    "4. Routing logic (should_continue function)\n"
    "5. Key implementation notes\n\n"
    "CRITICAL — TOOLS_JSON must contain tools that DIRECTLY serve the task:\n"
    "- Think: what real-world actions does this agent need to perform?\n"
    "- Examples for 'deepfake detector': analyze_image_metadata, check_face_consistency, search_deepfake_databases\n"
    "- Examples for 'weather agent': get_current_weather, get_forecast\n"
    "- Examples for 'research agent': web_search, fetch_webpage, summarize_text\n"
    "- NEVER use AgentForge framework tools as agent tools: search_documentation, "
    "generate_mermaid_diagram, read_file, list_output_files are INTERNAL tools — "
    "they must NOT appear in TOOLS_JSON\n\n"
    "Embed at the end of your response:\n"
    'TOOLS_JSON: [{"name": "snake_case_tool_name", "purpose": "what it does and returns for this specific task"}, ...]\n\n'
    "Be specific. The Developer implements exactly what you specify."
)

MULTI_ARCH_PROMPT = (
    "You are a Multi-Agent Architect designing a real AI pipeline for a specific task.\n\n"
    "Design a production-ready LangGraph multi-agent pipeline for the task given in context.\n\n"
    "Produce:\n"
    "1. Mermaid diagram of all agents and connections\n"
    "2. For each agent: role, which tools it uses, system prompt summary\n"
    "3. Routing logic between agents (with clear termination)\n"
    "4. Shared State TypedDict fields\n"
    "5. Tool assignment: which agent calls which tools\n"
    "6. Key implementation notes\n\n"
    "Every tool-using agent needs its own ToolNode ReAct loop.\n"
    "Non-tool synthesis agents use plain llm.invoke().\n\n"
    "CRITICAL — TOOLS_JSON must contain tools that DIRECTLY serve the task:\n"
    "- Design STRICTLY SEQUENTIAL pipelines (e.g., Node A -> Node B -> Node C -> END).\n"
    "- DO NOT use parallel branching (e.g., A -> B and A -> C) as it causes LangGraph InvalidUpdateErrors on shared state.\n\n"
    "Every tool-using agent needs its own ToolNode ReAct loop.\n"
    "- Think: what real-world data or actions does this pipeline need?\n"
    "- Assign each tool to a specific specialist agent role\n"
    "- NEVER use AgentForge framework tools as agent tools: search_documentation, "
    "generate_mermaid_diagram, read_file, list_output_files are INTERNAL tools — "
    "they must NOT appear in TOOLS_JSON\n\n"
    "Embed at the end of your response:\n"
    'TOOLS_JSON: [{"name": "snake_case_tool_name", "purpose": "what it does and returns for this specific task"}, ...]\n\n'
    "Be specific. The Developer implements exactly what you specify."
)

# ── Constitutional Critic ─────────────────────────────────────────────────────
CONSTITUTIONAL_CRITIC_PROMPT = (
    "You are a Constitutional Code Critic for AgentForge v3.\n\n"
    "Score agent.py against this rubric (each dimension 0-2):\n\n"
    "  [llm_calls]     — ChatOpenAI instantiated AND llm.invoke()/llm_with_tools.invoke() called\n"
    "  [tools_import]  — 'from tools import' present, tools NOT redefined inline\n"
    "  [correctness]   — correct imports (langchain_core.*), valid StateGraph wiring\n"
    "  [robustness]    — _sanitize_messages used, routing has termination condition\n"
    "  [completeness]  — agent.py, run.py, requirements.txt all present\n\n"
    "verdict = PASS if total >= 6 AND llm_calls >= 1 AND tools_import >= 1\n\n"
    "Respond ONLY with JSON:\n"
    '{"llm_calls":2,"tools_import":2,"correctness":2,"robustness":1,"completeness":2,'
    '"total":9,"critical_issues":[],"suggestions":[],"verdict":"PASS"}'
)

# ── Fixer ─────────────────────────────────────────────────────────────────────
FIXER_PROMPT = (
    "You are an expert LangGraph developer fixing failing agent code.\n\n"
    "Read the test report + AST report + critic feedback in context.\n"
    "Produce corrected agent.py, run.py, requirements.txt as a single raw JSON.\n"
    "DO NOT include tools.py — the system manages it.\n\n"
    "COMMON FAILURES:\n\n"
    "1. ValueError: Graph must have an entrypoint\n"
    "   → Add: workflow.add_edge(START, 'first_node_name')\n"
    "   → Ensure: from langgraph.graph import StateGraph, END, START\n\n"
    "2. ValueError: Found edge starting at unknown node 'X'\n"
    "   → String in add_edge('X') must EXACTLY match add_node('X', fn)\n\n"
    "3. BadRequestError: tool_call_ids did not have response messages\n"
    "   → Add _sanitize_messages() call before every llm_with_tools.invoke()\n\n"
    "4. ChatOpenAI never instantiated / no llm.invoke()\n"
    "   → Every node must call llm_with_tools.invoke([system] + safe_msgs)\n\n"
    "5. ModuleNotFoundError: from tools import\n"
    "   → Ensure agent.py has: from tools import <names>, ALL_TOOLS\n\n"
    "RULES:\n"
    "- Only langchain_core.messages imports\n"
    "- import operator; use Annotated[List[BaseMessage], operator.add]\n"
    "- requirements.txt: langgraph, langchain-openai, langchain-core, openai + tool packages\n\n"
    "Output ONLY the JSON."
)

# ── Tester ────────────────────────────────────────────────────────────────────
TESTER_PROMPT = (
    "You are a QA Tester for generated LangGraph agent code.\n\n"
    "Replace AGENT_NAME with the actual name from context.\n\n"
    "Steps IN ORDER:\n"
    "1. run_ast_validation on output/AGENT_NAME/agent.py\n"
    "2. run_syntax_check on output/AGENT_NAME/tools.py\n"
    "3. install_requirements from output/AGENT_NAME/requirements.txt\n"
    "4. execute_script output/AGENT_NAME/run.py\n"
    "5. analyze_logs\n\n"
    "PASS: syntax clean, imports resolve, no structural runtime errors.\n"
    "FAIL: SyntaxError, ImportError, ModuleNotFoundError, AttributeError.\n"
    "NOT failures: AuthenticationError, RateLimitError (no real key — expected).\n\n"
    "End with:\n  VERDICT: PASS\n  VERDICT: FAIL - <reason>"
)


# ── Developer (dynamic — depends on state) ────────────────────────────────────
def build_developer_prompt(state: dict) -> str:
    tool_names  = state.get("tool_names",   "ALL_TOOLS")
    tool_pkgs   = state.get("tool_packages", "[]")
    agent_name  = state.get("agent_name",    "my_agent")
    tools_import = f"from tools import {tool_names}, ALL_TOOLS"

    single_skeleton = (
        "import operator, os\n"
        "from typing import Annotated, List, TypedDict\n"
        "from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, BaseMessage\n"
        "from langchain_openai import ChatOpenAI\n"
        "from langgraph.graph import StateGraph, END, START\n"
        "from langgraph.prebuilt import ToolNode\n"
        "from langgraph.checkpoint.memory import MemorySaver\n"
        f"{tools_import}\n\n"
        "llm = ChatOpenAI(openai_api_key=os.environ.get('OPENAI_API_KEY',''), model='gpt-4o', temperature=0)\n"
        "llm_with_tools = llm.bind_tools(ALL_TOOLS)\n\n"
        "# _sanitize_messages() is auto-injected by the system — use it!\n\n"
        "class AgentState(TypedDict):\n"
        "    messages: Annotated[List[BaseMessage], operator.add]\n\n"
        "def agent_node(state: AgentState) -> dict:\n"
        "    system = SystemMessage(content='You are a helpful assistant.')\n"
        "    safe = _sanitize_messages(state['messages'])\n"
        "    resp = llm_with_tools.invoke([system] + safe)\n"
        "    return {'messages': [resp]}\n\n"
        "def should_continue(state: AgentState) -> str:\n"
        "    last = state['messages'][-1]\n"
        "    return 'tools' if (hasattr(last, 'tool_calls') and last.tool_calls) else END\n\n"
        "workflow = StateGraph(AgentState)\n"
        "workflow.add_node('agent', agent_node)\n"
        "workflow.add_node('tools', ToolNode(ALL_TOOLS))\n"
        "workflow.add_edge(START, 'agent')\n"
        "workflow.add_conditional_edges('agent', should_continue, {'tools': 'tools', END: END})\n"
        "workflow.add_edge('tools', 'agent')\n"
        "app = workflow.compile(checkpointer=MemorySaver())\n"
    )

    return (
        "You are an expert LangGraph developer writing a REAL, production-ready agent.\n\n"

        "━━━ RULE 1: IMPORT TOOLS FROM tools.py ━━━\n"
        f"Your agent.py MUST contain: {tools_import}\n"
        "DO NOT redefine tools inline. DO NOT include tools.py in your JSON output.\n\n"

        "━━━ RULE 2: SANITIZE MESSAGES BEFORE EVERY LLM CALL ━━━\n"
        "_sanitize_messages() is auto-injected. Use it:\n"
        "  safe = _sanitize_messages(state['messages'])\n"
        "  resp = llm_with_tools.invoke([system] + safe)\n\n"

        "━━━ RULE 3: EVERY llm_with_tools NODE NEEDS A ToolNode LOOP ━━━\n"
        "  node → (has tool_calls?) → ToolNode → back to same node\n"
        "Sequential pipeline nodes should use plain llm.invoke().\n\n"

        "━━━ SKELETON ━━━\n"
        f"{single_skeleton}\n\n"

        "━━━ OUTPUT FORMAT ━━━\n"
        "Single raw JSON. Use the actual agent_name from context.\n"
        f'Files: "output/{agent_name}/agent.py", "output/{agent_name}/requirements.txt", '
        f'"output/{agent_name}/run.py", "output/{agent_name}/README.md"\n'
        "DO NOT include tools.py.\n\n"

        "━━━ REQUIREMENTS.TXT ━━━\n"
        "Always include: langgraph, langchain-openai, langchain-core, openai\n"
        f"Also include: {tool_pkgs}\n"
        "NEVER include bare 'langchain'.\n\n"

        "━━━ BANS ━━━\n"
        "✗ No pass / ... / stubs\n"
        "✗ No class-based StateGraph\n"
        "✗ No langchain.schema imports\n\n"

        "Output ONLY the JSON."
    )

Overwriting prompts.py


In [ ]:
%%writefile state.py
"""state.py — Shared AgentState TypedDict for the LangGraph pipeline."""
from typing import Annotated, List, TypedDict
from langchain_core.messages import BaseMessage


class AgentState(TypedDict):
    messages:           Annotated[List[BaseMessage], lambda x, y: x + y]
    task:               str
    agent_name:         str
    architecture_type:  str          # "single" | "multi"
    architecture_spec:  str
    # Tool pipeline
    tool_specs:         str          # JSON list [{name, purpose}]
    tools_py:           str          # generated tools.py source
    tool_names:         str          # comma-separated tool names
    tool_packages:      str          # JSON list of pip packages
    # Code generation
    code_files:         str          # JSON map filepath -> content
    # Quality gates
    ast_report:         str
    critic_score:       int
    critic_feedback:    str
    test_report:        str
    fix_attempts:       int
    iteration_count:    int
    output_dir:         str
    fingerprint:        str
    final_summary:      str


Overwriting state.py


In [ ]:
%%writefile agent_template.py
"""
agent_template.py — Hybrid agent assembler.

SPLIT RESPONSIBILITY:
  Template (this file):
    - All imports (correct, pinned)
    - _sanitize_messages() definition
    - AgentState TypedDict with messages + iteration_count
    - LLM + llm_with_tools instantiation
    - StateGraph wiring (add_node, add_edge, compile)
    - run.py invocation
    - requirements.txt
    - README.md

  LLM-generated (via generate_nodes()):
    - Node function bodies  (injected verbatim into the skeleton)
    - System prompt strings (injected into node bodies)
    - Extra state fields    (injected into AgentState)
    - Routing condition bodies

  Surgical repair (via repair_agent_py()):
    - AST validation after assembly
    - Fix missing docstrings, wrong imports, node name mismatches, missing START edge
    - Re-inject _sanitize_messages if somehow missing
"""

import ast
import json
import re
import textwrap
from typing import Any

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

from config import OPENAI_API_KEY, invoke_with_backoff

# ─────────────────────────────────────────────────────────────────────────────
# NOT-PIP: packages that look like package names but can't be pip-installed
# ─────────────────────────────────────────────────────────────────────────────
_NOT_PIP = {
    "os", "sys", "re", "json", "io", "pathlib", "subprocess", "stdlib",
    "collections", "itertools", "functools", "typing", "datetime", "math",
    "random", "hashlib", "base64", "urllib", "socket", "threading",
    "multiprocessing", "logging", "copy", "time", "struct", "csv",
}

# ─────────────────────────────────────────────────────────────────────────────
# PART 1: LLM generates node bodies, prompts, state fields
# ─────────────────────────────────────────────────────────────────────────────

_NODE_GEN_SYSTEM = """You are an expert LangGraph developer generating agent node functions.

You will be given:
- A task description
- Available tools (already imported from tools.py)
- Architecture specification
- Number and role of nodes needed

Generate the node functions and supporting config. Rules:
1. Every node function: def node_name(state: AgentState) -> dict
2. Always call _sanitize_messages(state["messages"]) before llm.invoke()
3. Always return {"messages": [resp], "iteration_count": state.get("iteration_count",0)+1}
4. CRITICAL — per-node tool binding:
   - Each node that uses tools must bind its OWN subset of tools, not ALL_TOOLS
   - Use: node_llm = _llm.bind_tools([tool1, tool2]) inside the node, or use pre-bound _llm_with_tools only if ALL tools are relevant
   - Multi-agent: each specialist node binds only the tools relevant to its role
   - Example: search_node binds [web_search], analysis_node binds [parse_html], synthesis_node uses plain _llm
5. Use _llm_with_tools.invoke() if node uses ALL tools, or create node_llm = _llm.bind_tools([specific_tools]) for specialist nodes
6. System prompts must be task-specific and detailed
7. CRITICAL — The router MUST ONLY check `last.tool_calls`. NEVER check custom state fields (causes KeyError). Return "tools" or END.
# ── Surgical repair of router body ──8. For extra state fields use only simple types: str, int, float, List[str]
9. NEVER import anything — all imports are already at module level
10. NEVER define _sanitize_messages — it is already defined

For multi-agent, use this per-node binding pattern:
  def search_node(state: AgentState) -> dict:
      _node_llm = _llm.bind_tools([web_search, fetch_webpage])  # only relevant tools
      safe = _sanitize_messages(state["messages"])
      resp = _node_llm.invoke([SystemMessage(content=_SYSTEM_PROMPT_SEARCH)] + safe)
      return {"messages": [resp], "iteration_count": state.get("iteration_count", 0) + 1}

Respond ONLY with JSON (no markdown):
{
  "nodes": [
    {
      "name": "agent_node",
      "uses_tools": true,
      "tool_subset": ["web_search"],
      "system_prompt": "You are a ...",
      "body": "    _node_llm = _llm.bind_tools([web_search])\\n    system = SystemMessage(content=_SYSTEM_PROMPT_AGENT_NODE)\\n    safe = _sanitize_messages(state[\\"messages\\"])\\n    resp = _node_llm.invoke([system] + safe)\\n    return {\\"messages\\": [resp], \\"iteration_count\\": state.get(\\"iteration_count\\", 0) + 1}"
    }
  ],
  "router": {
    "name": "should_continue",
    "body": "    last = state[\\"messages\\"][-1]\\n    if state.get(\\"iteration_count\\", 0) >= 10:\\n        return END\\n    if hasattr(last, \\"tool_calls\\") and last.tool_calls:\\n        return \\"tools\\"\\n    return END"
  },
  "extra_state_fields": {"result": "str"},
  "graph_edges": [
    "workflow.add_edge(START, 'agent_node')",
    "workflow.add_conditional_edges('agent_node', should_continue, {'tools': 'tools', END: END})",
    "workflow.add_edge('tools', 'agent_node')"
  ]
}
"""


def generate_nodes(
    task: str,
    tool_names: list[str],
    arch_spec: str,
    arch_type: str,
) -> dict:
    """
    Ask LLM to generate node function bodies, system prompts, extra state fields,
    and graph edges. Each node binds only its relevant tool subset.
    """
    tools_str = ", ".join(tool_names) if tool_names else "none"

    if arch_type == "single":
        node_guidance = (
            "Generate ONE agent node named 'agent_node'.\n"
            "If there are tools, bind ALL of them: _node_llm = _llm.bind_tools(ALL_TOOLS)\n"
            "Generate ONE router named 'should_continue'.\n"
            "Graph: START → agent_node ↔ tools → END"
        )
    else:
        # Build per-tool guidance for multi-agent
        tool_list = "\n".join(f"  - {t}" for t in tool_names) if tool_names else "  - none"
        node_guidance = (
            f"Based on the architecture spec, generate 2-4 specialized nodes.\n"
            f"Available tools to distribute among nodes:\n{tool_list}\n\n"
            "For each tool-using node:\n"
            "  - Assign only the tools relevant to that node's role\n"
            "  - Bind them with: _node_llm = _llm.bind_tools([tool1, tool2])\n"
            "  - Each tool-using node needs its own ToolNode with its specific tools\n"
            "    Example: workflow.add_node('search_tools', ToolNode([web_search]))\n"
            "Pure synthesis/aggregation nodes use plain _llm (no tool binding).\n"
            "Wire them in sequence with proper conditional edges per tool-using node."
        )

    prompt = (
        f"Task: {task}\n\n"
        f"Architecture type: {arch_type}\n"
        f"Available tools (already imported: from tools import {tools_str}, ALL_TOOLS): {tools_str}\n\n"
        f"Architecture spec:\n{arch_spec[:1500]}\n\n"
        f"Instructions:\n{node_guidance}\n\n"
        "Generate the node functions. All code goes into the 'body' field as a string.\n"
        "The body is indented Python code (4 spaces) — the def line is added by the assembler.\n"
        "For per-node tool binding, put the bind_tools call as the FIRST line of the body.\n"
        "Respond with ONLY the JSON."
    )

    llm = ChatOpenAI(openai_api_key=OPENAI_API_KEY, model="gpt-4o", temperature=0)
    try:
        resp    = invoke_with_backoff(llm, [
            SystemMessage(content=_NODE_GEN_SYSTEM),
            HumanMessage(content=prompt),
        ])
        cleaned = re.sub(r"```(?:json)?", "", resp.content or "").replace("```", "").strip()
        return json.loads(cleaned)
    except Exception as e:
        print(f"  [NodeGen] LLM failed ({e}), using fallback spec")
        return _fallback_spec(task, tool_names)


def _fallback_spec(task: str, tool_names: list[str]) -> dict:
    """Minimal correct node spec when LLM fails."""
    has_tools = bool(tool_names)
    prompt_text = f"You are a helpful AI assistant for: {task}. Use your tools to help the user."
    node_body = (
        '    system = SystemMessage(content=_SYSTEM_PROMPT_AGENT)\n'
        '    safe = _sanitize_messages(state["messages"])\n'
        '    resp = _llm_with_tools.invoke([system] + safe)\n'
        '    return {"messages": [resp], "iteration_count": state.get("iteration_count", 0) + 1}'
    )
    router_body = (
        '    last = state["messages"][-1]\n'
        '    if state.get("iteration_count", 0) >= 10:\n'
        '        return END\n'
        '    if hasattr(last, "tool_calls") and last.tool_calls:\n'
        '        return "tools"\n'
        '    return END'
    )
    return {
        "nodes": [{"name": "agent_node", "uses_tools": has_tools,
                   "system_prompt": prompt_text, "body": node_body}],
        "router": {"name": "should_continue", "body": router_body},
        "extra_state_fields": {},
        "graph_edges": [
            "workflow.add_edge(START, 'agent_node')",
            "workflow.add_conditional_edges('agent_node', should_continue, {'tools': 'tools', END: END})",
            "workflow.add_edge('tools', 'agent_node')",
        ],
    }


# ─────────────────────────────────────────────────────────────────────────────
# PART 2: Assembler — splice LLM parts into the fixed template skeleton
# ─────────────────────────────────────────────────────────────────────────────

def assemble_agent_py(
    tool_names: list[str],
    node_spec: dict,
) -> str:
    """
    Assemble the final agent.py by splicing LLM-generated node bodies
    into a fixed, correct template skeleton.
    """
    nodes           = node_spec.get("nodes", [])
    router          = node_spec.get("router", {})
    extra_fields    = node_spec.get("extra_state_fields", {})
    graph_edges_raw = node_spec.get("graph_edges", [])

    # ── Tools import ──────────────────────────────────────────────────────────
    if tool_names:
        tools_import = f"from tools import {', '.join(tool_names)}, ALL_TOOLS"
    else:
        tools_import = "ALL_TOOLS = []  # no tools for this task"

    # ── Extra state fields ────────────────────────────────────────────────────
    extra_fields_src = ""
    for fname, ftype in (extra_fields or {}).items():
        clean_name = re.sub(r"[^a-z0-9_]", "_", str(fname).lower())
        clean_type = str(ftype) if isinstance(ftype, str) else "str"
        if not re.match(r'^[A-Za-z_\[\], ]+$', clean_type):
            clean_type = "str"
        extra_fields_src += f"    {clean_name}: {clean_type}\n"

    # ── System prompt constants ───────────────────────────────────────────────
    prompt_constants = []
    for node in nodes:
        const_name = f"_SYSTEM_PROMPT_{node['name'].upper()}"
        raw_prompt = node.get("system_prompt", "You are a helpful AI assistant.")
        safe = raw_prompt.replace("\\", "\\\\").replace('"""', '\\"\\"\\"')
        prompt_constants.append(f'{const_name} = """{safe}"""')

    # ── Node function bodies ──────────────────────────────────────────────────
    node_funcs = []
    for node in nodes:
        name        = node["name"]
        body        = node.get("body", "")
        uses_tools  = node.get("uses_tools", True)
        tool_subset = node.get("tool_subset", tool_names)  # per-node tool list
        const_name  = f"_SYSTEM_PROMPT_{name.upper()}"

        # ── Surgical repair of LLM-generated node body ────────────────────────

        # Repair 1: unescape literal \n from JSON
        body = body.replace("\\n", "\n")

        # Repair 2: normalize any hallucinated system prompt constant name
        body = re.sub(r"_SYSTEM_PROMPT_[A-Z0-9_]+", const_name, body)

        # Repair 3: inject system prompt assignment if missing
        if const_name not in body:
            body = f"    system = SystemMessage(content={const_name})\n" + body

        # Repair 4: enforce per-node tool binding if node uses tools
        # If LLM used _llm_with_tools globally but we have a specific subset, replace
        if uses_tools and tool_subset and tool_subset != tool_names:
            subset_str = "[" + ", ".join(tool_subset) + "]"
            # Replace global _llm_with_tools with per-node binding
            if "_llm_with_tools.invoke" in body and "_node_llm" not in body:
                body = (f"    _node_llm = _llm.bind_tools({subset_str})\n" + body)
                body = body.replace("_llm_with_tools.invoke", "_node_llm.invoke")
        elif uses_tools and tool_names and "_node_llm" not in body and "_llm_with_tools" not in body:
            # Node wants tools but has neither — inject bind_tools
            body = f"    _node_llm = _llm.bind_tools(ALL_TOOLS)\n" + body
            body = body.replace("_llm.invoke", "_node_llm.invoke")

        # Repair 5: force correct llm variant based on uses_tools
        if uses_tools and tool_names:
            # Must NOT use plain _llm for the primary invoke
            # (allow _llm in sub-calls but main invoke should use tool-bound version)
            pass  # per-node binding handles this above
        elif not uses_tools:
            # Pure synthesis node — must use plain _llm, not _llm_with_tools
            body = body.replace("_llm_with_tools.invoke", "_llm.invoke")
            body = body.replace("_node_llm.invoke", "_llm.invoke")

        # Repair 6: normalize indentation
        body = _normalize_indent(body, 4)

        # Repair 7: ensure _sanitize_messages is called
        if "_sanitize_messages" not in body:
            body = (
                f'    safe = _sanitize_messages(state["messages"])\n'
                + body.replace('state["messages"]', 'safe')
                      .replace("state['messages']", "safe")
            )

        # Repair 8: ensure return statement exists
        if "return {" not in body and "return{" not in body:
            body += (
                '\n    return {"messages": [resp], '
                '"iteration_count": state.get("iteration_count", 0) + 1}'
            )

        func = (
            f"def {name}(state: AgentState) -> dict:\n"
            f'    """Node: {name} — task-specific logic."""\n'
            f"{body}\n"
        )
        node_funcs.append(func)

    # ── Router function ───────────────────────────────────────────────────────
    router_name = router.get("name", "should_continue")
    router_body = router.get("body", "")
    router_body = router_body.replace("\\n", "\n")
    router_body = _normalize_indent(router_body, 4)

    # ── Surgical repair of router body ────────────────────────────────────────
    # Repair R0: Aggressive fallback if LLM hallucinates custom state checks
    if "tool_calls" not in router_body:
        router_body = (
            '    last = state["messages"][-1]\n'
            '    if hasattr(last, "tool_calls") and last.tool_calls:\n'
            '        return "tools"\n'
            '    return END'
        )
    # Repair R1: replace bare boolean returns with proper string/END returns
    # Pattern: `return hasattr(...) and ...` or `return last.tool_calls` — returns bool
    router_body = re.sub(
        r"return\s+hasattr\s*\([^)]+\)\s+and\s+[^\n]+",
        'if hasattr(state["messages"][-1], "tool_calls") and state["messages"][-1].tool_calls:\n'
        '        return "tools"\n'
        '    return END',
        router_body,
    )
    # Pattern: `return True` / `return False` at top level in a router
    router_body = re.sub(r'\breturn\s+True\b',  'return "tools"', router_body)
    router_body = re.sub(r'\breturn\s+False\b', 'return END',     router_body)

    # Repair R2: ensure there is at least one END return
    if "return END" not in router_body and "return end" not in router_body.lower():
        router_body += "\n    return END"

    # Repair R3: if router is empty after unescape/normalise, use safe default
    if not router_body.strip():
        router_body = (
            '    last = state["messages"][-1]\n'
            '    if state.get("iteration_count", 0) >= 10:\n'
            '        return END\n'
            '    if hasattr(last, "tool_calls") and last.tool_calls:\n'
            '        return "tools"\n'
            '    return END'
        )

    router_func = (
        f"def {router_name}(state: AgentState) -> str:\n"
        f'    """Routing function — returns next node name or END."""\n'
        f"{router_body}\n"
    )

    # ── Graph wiring ──────────────────────────────────────────────────────────
    tool_node_registrations = []
    registered_tool_nodes: set[str] = set()

    # Register agent nodes
    for node in nodes:
        tool_node_registrations.append(f"workflow.add_node('{node['name']}', {node['name']})")

    # Register ToolNodes — per-node subsets for multi-agent, shared ALL_TOOLS for single
    is_multi = len(nodes) > 1
    for node in nodes:
        if not node.get("uses_tools", True) or not tool_names:
            continue
        tool_subset = node.get("tool_subset") or tool_names

        if not is_multi or set(tool_subset) == set(tool_names):
            # Single agent or node using all tools — one shared 'tools' ToolNode
            if "tools" not in registered_tool_nodes:
                tool_node_registrations.append("workflow.add_node('tools', ToolNode(ALL_TOOLS))")
                registered_tool_nodes.add("tools")
        else:
            # Multi-agent — dedicated ToolNode per specialist node
            tn_name    = f"{node['name']}_tools"
            subset_str = "[" + ", ".join(tool_subset) + "]"
            if tn_name not in registered_tool_nodes:
                tool_node_registrations.append(
                    f"workflow.add_node('{tn_name}', ToolNode({subset_str}))"
                )
                registered_tool_nodes.add(tn_name)

    # Sanitize LLM-generated graph edges
    graph_edges = []
    for edge in graph_edges_raw:
        e = edge.strip()
        # Repair: replace bare 'tools' refs with per-node tool node names when needed
        if is_multi:
            for node in nodes:
                tn_name = f"{node['name']}_tools"
                if tn_name in registered_tool_nodes:
                    # Fix edges that route from this node to 'tools'
                    e = re.sub(
                        rf"(add_conditional_edges\s*\(\s*['\"]){re.escape(node['name'])}(['\"].*?['\"])tools(['\"])",
                        lambda m, tn=tn_name: m.group(0).replace("'tools'", f"'{tn}'").replace('"tools"', f'"{tn}"'),
                        e
                    )
        else:
            # SINGLE AGENT: Fix hallucinated edge mappings where 'tools' maps back to the agent
            # E.g., {'tools': 'agent_node'} -> {'tools': 'tools'}
            e = re.sub(r"(['\"]tools['\"]\s*:\s*['\"])[^'\"]+(['\"])", r"\g<1>tools\2", e)

        graph_edges.append(e)

    # Ensure START edge exists
    has_start = any("START" in e or "__start__" in e for e in graph_edges)
    if not has_start and nodes:
        first_node = nodes[0]["name"]
        graph_edges.insert(0, f"workflow.add_edge(START, '{first_node}')")

    # ── Assemble ──────────────────────────────────────────────────────────────
    parts = [
        "# agent.py — Generated by AgentForge v3",
        "# Skeleton is fixed. Node bodies and prompts are LLM-generated.",
        "import operator",
        "import os",
        "from typing import Annotated, List, TypedDict",
        "",
        "from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage",
        "from langchain_openai import ChatOpenAI",
        "from langgraph.checkpoint.memory import MemorySaver",
        "from langgraph.graph import END, START, StateGraph",
        "from langgraph.prebuilt import ToolNode",
        "",
        tools_import,
        "",
        "",
        "# ── Sanitize helper (FIXED — do not modify) ──────────────────────────────",
        _SANITIZE_MESSAGES_SRC,
        "",
        "",
        "# ── State ────────────────────────────────────────────────────────────────",
        "class AgentState(TypedDict):",
        "    messages:        Annotated[List[BaseMessage], operator.add]",
        "    iteration_count: Annotated[int, lambda x, y: max(x, y)]",
    ]
    if extra_fields_src:
        parts.append(extra_fields_src.rstrip())
    parts += [
        "",
        "",
        "# ── LLM setup ────────────────────────────────────────────────────────────",
        "_llm = ChatOpenAI(",
        '    openai_api_key=os.environ.get("OPENAI_API_KEY", ""),',
        '    model="gpt-4o",',
        "    temperature=0,",
        ")",
        "_llm_with_tools = _llm.bind_tools(ALL_TOOLS) if ALL_TOOLS else _llm",
        "",
        "",
        "# ── System prompts (LLM-generated) ───────────────────────────────────────",
    ]
    parts += prompt_constants
    parts += [
        "",
        "",
        "# ── Node functions (LLM-generated bodies, fixed signatures) ──────────────",
    ]
    parts += node_funcs
    parts += [
        "",
        "# ── Router ───────────────────────────────────────────────────────────────",
        router_func,
        "",
        "# ── Graph assembly (FIXED) ───────────────────────────────────────────────",
        "workflow = StateGraph(AgentState)",
    ]
    parts += tool_node_registrations
    parts += graph_edges
    parts += [
        "",
        "app = workflow.compile(checkpointer=MemorySaver())",
    ]

    return "\n".join(parts) + "\n"


# ─────────────────────────────────────────────────────────────────────────────
# PART 3: Surgical repair — fix AST issues in assembled agent.py
# ─────────────────────────────────────────────────────────────────────────────

def repair_agent_py(source: str) -> tuple[str, list[str]]:
    """
    Run surgical repairs on the assembled agent.py.
    Returns (repaired_source, list_of_repairs_applied).

    Repairs:
    1. Wrong import paths (langchain.schema → langchain_core.messages etc.)
    2. Duplicate/conflicting imports
    3. Missing _sanitize_messages (re-inject)
    4. Missing START edge
    5. Node name mismatches between add_node and add_edge
    6. Syntax check — if still broken after repairs, replace with fallback
    """
    repairs = []

    # ── Repair 0: router boolean returns → string/END ─────────────────────────
    # KeyError: True / KeyError: False happens when a router returns a boolean.
    # LLM sometimes writes: `return hasattr(last, "tool_calls") and last.tool_calls`
    # which returns True/False instead of "tools"/END.
    source = re.sub(
        r"(return\s+)hasattr\s*\([^)]+\)\s+and\s+[^\n]+",
        lambda m: (
            'if hasattr(state["messages"][-1], "tool_calls") and '
            'state["messages"][-1].tool_calls:\n        return "tools"\n    return END'
        ),
        source,
    )
    # Bare `return True` / `return False` inside routing functions
    # Only fix inside functions that have a -> str annotation (routers)
    lines_src = source.splitlines()
    in_router = False
    fixed_lines = []
    for line in lines_src:
        if re.search(r"def\s+\w+\s*\(state.*\)\s*->\s*str\s*:", line):
            in_router = True
        elif re.match(r"^def\s", line):
            in_router = False
        if in_router:
            if re.match(r"\s*return\s+True\b", line):
                line = line.replace("return True", 'return "tools"')
                repairs.append("Fixed router: return True → return 'tools'")
            elif re.match(r"\s*return\s+False\b", line):
                line = line.replace("return False", "return END")
                repairs.append("Fixed router: return False → return END")
        fixed_lines.append(line)
    source = "\n".join(fixed_lines)

    # ── Repair 1: wrong import paths ─────────────────────────────────────────
    bad_import_map = {
        r"from\s+langchain\.schema\s+import":         "from langchain_core.messages import",
        r"from\s+langchain_core\.schema\s+import":    "from langchain_core.messages import",
        r"from\s+langchain\.chat_models\s+import":    "from langchain_openai import",
        r"from\s+langchain\.llms\s+import":           "from langchain_openai import",
        r"from\s+langchain_core\.llms\s+import":      "from langchain_openai import",
        r"from\s+langchain_core\.prompts\s+import\s+SystemMessage":
            "from langchain_core.messages import SystemMessage",
    }
    for pattern, replacement in bad_import_map.items():
        new_source = re.sub(pattern, replacement, source)
        if new_source != source:
            repairs.append(f"Fixed import: {pattern.split(chr(92)+'s')[0]}")
            source = new_source

    # ── Repair 2: remove bare 'langchain' imports (old monolith) ─────────────
    lines = source.splitlines()
    filtered = []
    for line in lines:
        m = re.match(r'\s*(?:from|import)\s+([\w\.]+)', line)
        if m:
            mod = m.group(1)
            # Ban exactly "langchain" but not "langchain_core" or "langchain_openai"
            if mod == "langchain" or mod.startswith("langchain."):
                repairs.append(f"Removed banned import: {line.strip()}")
                continue
        filtered.append(line)
    source = "\n".join(filtered)

    # ── Repair 3: ensure _sanitize_messages is present ───────────────────────
    if "_sanitize_messages" not in source:
        # Find last top-level import and inject after it
        lines = source.splitlines(keepends=True)
        last_import = 0
        for i, line in enumerate(lines):
            if line.lstrip().startswith(("import ", "from ")):
                last_import = i
        lines.insert(last_import + 1, "\n" + _SANITIZE_MESSAGES_SRC + "\n\n")
        source = "".join(lines)
        repairs.append("Re-injected _sanitize_messages")

    # ── Repair 4: missing START edge ─────────────────────────────────────────
    has_start = bool(re.search(r'add_edge\s*\(\s*START\b', source))
    if not has_start:
        # Find first add_node that isn't tools
        m = re.search(r"workflow\.add_node\s*\(\s*['\"]([^'\"]+)['\"]", source)
        if m:
            first = m.group(1)
            if first != "tools":
                source = re.sub(
                    r"(app\s*=\s*workflow\.compile)",
                    f"workflow.add_edge(START, '{first}')  # auto-repair\n\\1",
                    source, count=1,
                )
                repairs.append(f"Injected START edge to '{first}'")

    # ── Repair 5: node name consistency ──────────────────────────────────────
    try:
        tree = ast.parse(source)
        registered = {}
        fn_to_node  = {}
        for node in ast.walk(tree):
            if not isinstance(node, ast.Call): continue
            if getattr(node.func, 'attr', '') == 'add_node' and len(node.args) >= 2:
                na, fa = node.args[0], node.args[1]
                if isinstance(na, ast.Constant) and isinstance(na.value, str):
                    nn = na.value
                    fn = fa.id if isinstance(fa, ast.Name) else ast.unparse(fa)
                    registered[nn] = fn
                    fn_to_node[fn]  = nn

        reg_set = set(registered) | {'__start__', '__end__', 'START', 'END'}

        def best_match(bad: str) -> str | None:
            if bad in fn_to_node:            return fn_to_node[bad]
            if bad.endswith('_node'):
                c = bad[:-5]
                if c in reg_set:             return c
            if bad + '_node' in reg_set:     return bad + '_node'
            starts = [r for r in reg_set if r.startswith(bad) and r != bad]
            if len(starts) == 1:             return starts[0]
            return None

        edge_lines = []
        other_lines = []
        for line in source.splitlines():
            if 'add_edge' in line or 'add_conditional_edges' in line:
                def replace_name(m: re.Match) -> str:
                    q, name = m.group(1), m.group(2)
                    if name in reg_set: return m.group(0)
                    fix = best_match(name)
                    if fix:
                        repairs.append(f"Node name: '{name}' → '{fix}'")
                        return f"{q}{fix}{q}"
                    return m.group(0)
                line = re.sub(r'(["\'])([^"\']+)\1', replace_name, line)
            other_lines.append(line)
        source = "\n".join(other_lines)
    except SyntaxError:
        pass  # will be caught in syntax check below

    # ── Repair 6: syntax check — if broken, note it ───────────────────────────
    try:
        ast.parse(source)
    except SyntaxError as e:
        repairs.append(f"WARNING: syntax error remains after repair: {e}")

    return source, repairs


# ─────────────────────────────────────────────────────────────────────────────
# FIXED template strings
# ─────────────────────────────────────────────────────────────────────────────

_SANITIZE_MESSAGES_SRC = textwrap.dedent('''\
    def _sanitize_messages(messages: list) -> list:
        """Remove unresolved tool_calls and orphaned ToolMessages (prevents OpenAI 400)."""
        from langchain_core.messages import AIMessage, ToolMessage
        covered = {m.tool_call_id for m in messages if isinstance(m, ToolMessage)}
        safe = []
        i = 0
        while i < len(messages):
            msg = messages[i]
            if isinstance(msg, AIMessage) and getattr(msg, "tool_calls", None):
                call_ids = {tc["id"] for tc in msg.tool_calls}
                if call_ids.issubset(covered):
                    safe.append(msg)
                    i += 1
                    collected: set = set()
                    while i < len(messages) and isinstance(messages[i], ToolMessage):
                        tm = messages[i]
                        if tm.tool_call_id in call_ids:
                            safe.append(tm)
                            collected.add(tm.tool_call_id)
                        i += 1
                    if not call_ids.issubset(collected):
                        while safe and isinstance(safe[-1], ToolMessage):
                            safe.pop()
                        safe[-1] = AIMessage(content=safe[-1].content or "(processing)")
                else:
                    safe.append(AIMessage(content=msg.content or "(processing)"))
                    i += 1
            elif isinstance(msg, ToolMessage):
                i += 1
            else:
                safe.append(msg)
                i += 1
        return safe''')


def _normalize_indent(code: str, spaces: int = 4) -> str:
    """Ensure all non-empty lines start with at least `spaces` spaces."""
    pad = " " * spaces
    result = []
    for line in code.splitlines():
        if line.strip():
            # Count existing indent
            existing = len(line) - len(line.lstrip())
            if existing < spaces:
                result.append(pad + line.lstrip())
            else:
                result.append(line)
        else:
            result.append("")
    return "\n".join(result)


# ─────────────────────────────────────────────────────────────────────────────
# PART 4: Fixed file builders
# ─────────────────────────────────────────────────────────────────────────────

def build_run_py(agent_name: str) -> str:
    """Guaranteed-correct run.py — never touches LLM output."""
    return textwrap.dedent(f'''\
        # run.py — Generated by AgentForge v3 (fixed template)
        import sys
        import os

        _AGENT_DIR = os.path.abspath(os.path.dirname(__file__))
        if _AGENT_DIR not in sys.path:
            sys.path.insert(0, _AGENT_DIR)

        from agent import app
        from langchain_core.messages import HumanMessage


        def run(query: str = "Hello, what can you help me with?") -> list:
            """Run the {agent_name} agent and stream results."""
            config        = {{"configurable": {{"thread_id": "run_1"}}}}
            initial_state = {{"messages": [HumanMessage(content=query)], "iteration_count": 0}}
            print(f"Running agent: {{query!r}}\\n")
            seen = []
            for event in app.stream(initial_state, config=config):
                for node_name, update in event.items():
                    if isinstance(update, dict) and "messages" in update:
                        for msg in update["messages"]:
                            text = getattr(msg, "content", "")
                            if text and text not in seen:
                                print(f"[{{node_name}}] {{text[:500]}}")
                                seen.append(text)
            print("\\nAgent run complete.")
            return seen


        if __name__ == "__main__":
            query = " ".join(sys.argv[1:]) or "Hello, what can you help me with?"
            run(query)
    ''')


def build_requirements_txt(tool_packages: list[str]) -> str:
    """Clean requirements.txt — no stdlib packages."""
    base   = ["langgraph", "langchain-openai", "langchain-core", "openai"]
    extras = sorted({
        p for p in tool_packages
        if p.lower() not in _NOT_PIP and p not in base and p.strip()
    })
    return "\n".join(base + extras) + "\n"


def build_readme(agent_name: str, description: str, tool_names: list[str], fingerprint: str = "N/A") -> str:
    """README.md."""
    tools_section = "\n".join(f"- `{t}`" for t in tool_names) if tool_names else "- No external tools"
    return textwrap.dedent(f"""\
        # {agent_name}

        {description}

        ## Tools
        {tools_section}

        ## Usage
        ```bash
        cd output/{agent_name}
        export OPENAI_API_KEY=your_key_here
        python run.py "Your query here"
        ```

        ## Fingerprint
        `{fingerprint}`
    """)

Overwriting agent_template.py


In [ ]:
%%writefile tool_gen.py
"""tool_gen.py — Generates real, task-specific @tool implementations from scratch."""
import ast
import json
import re
import textwrap

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

from config import OPENAI_API_KEY, invoke_with_backoff

# ── Package names that the LLM confuses for tool names ───────────────────────
# If the Tool Planner returns these, we remap them to semantic tool names.
_PACKAGE_TO_SEMANTIC = {
    "requests":       "http_request",
    "beautifulsoup4": "parse_html",
    "bs4":            "parse_html",
    "duckduckgo_search": "web_search",
    "duckduckgo-search": "web_search",
    "ddg":            "web_search",
    "stdlib":         "file_operations",
    "os":             "file_operations",
    "json":           "json_parser",
    "re":             "regex_search",
    "subprocess":     "run_command",
    "pathlib":        "file_operations",
}

TOOL_GEN_SYSTEM_PROMPT = """You are an expert Python developer writing REAL, fully-implemented tool functions for LangGraph agents.

ABSOLUTE RULES:
1. Every function body must contain REAL working code — NEVER use `pass`, `...`, `raise NotImplementedError`, or stub strings.
2. Every @tool function MUST have a docstring immediately after the def line (ValueError crash if missing).
3. ALL imports go INSIDE the function body — the function must be self-contained.
4. Return type is always `str` — even error paths return a descriptive string.
5. Wrap ALL logic in try/except and return a human-readable error string on failure.
6. Use only: requests, beautifulsoup4, duckduckgo-search, stdlib.
7. For data retrieval without API keys: use DuckDuckGo (duckduckgo-search) or free public REST endpoints.
8. NEVER require private API keys.

CRITICAL NAMING RULE:
- The "name" field in the JSON is the EXACT Python function name. Use snake_case.
- The generated function def line MUST match the name field exactly.
- WRONG: name="web_search", def duckduckgo_search_tool(...)  ← MISMATCH = crash
- RIGHT: name="web_search", def web_search(...)              ← MATCH = works

REAL+MOCK PATTERN (REQUIRED for every tool):
Each tool must have TWO implementations inside its body:
  1. A REAL implementation that calls actual APIs/services
  2. A MOCK fallback that returns plausible synthetic data when the real call fails

Use this pattern:
@tool
def web_search(query: str) -> str:
    \"\"\"Search the web using DuckDuckGo and return top results.\"\"\"
    # ── REAL implementation ──────────────────────────────────────────────
    try:
        from duckduckgo_search import DDGS
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=5))
        if results:
            return "\\n".join(f"{r['title']}: {r['body']}" for r in results)
    except Exception as real_err:
        pass  # fall through to mock
    # ── MOCK fallback (used when real implementation fails) ──────────────
    return (
        f"[MOCK] Web search results for '{query}':\\n"
        f"1. Overview of {query} — key concepts and recent developments.\\n"
        f"2. {query} in 2025 — latest trends and analysis.\\n"
        f"3. How {query} works — technical deep-dive."
    )

DuckDuckGo IMPORT: Use `from duckduckgo_search import DDGS` then `with DDGS() as ddgs: ddgs.text(...)`.
NEVER use `from duckduckgo_search import ddg` — that function does not exist.

Output a JSON array of tool objects:
[{"name": "exact_function_name", "description": "...", "packages": ["requests"], "code": "complete @tool source with real+mock pattern"}, ...]

No markdown. No text outside the JSON array.
"""


def _sanitize_tool_specs(tool_specs: list[dict]) -> list[dict]:
    """
    Remap package names to semantic tool names.
    If the Tool Planner returned 'requests' or 'beautifulsoup4' as tool names,
    replace them with meaningful names so the LLM generates useful tools.
    """
    sanitized = []
    seen_semantic = set()
    for spec in tool_specs:
        name    = spec.get("name", "").strip().lower()
        purpose = spec.get("purpose", "")
        # Remap if it's a known package name
        semantic = _PACKAGE_TO_SEMANTIC.get(name, name)
        # Avoid duplicates after remapping (e.g. two package names → same semantic)
        if semantic in seen_semantic:
            continue
        seen_semantic.add(semantic)
        sanitized.append({"name": semantic, "purpose": purpose or f"Execute {semantic} operations"})
    return sanitized


def _extract_actual_fn_name(code: str) -> str | None:
    """Extract the actual Python function name from @tool code using AST."""
    try:
        tree = ast.parse(code)
        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef):
                decs = [getattr(d, 'id', getattr(d, 'attr', '')) for d in node.decorator_list]
                if 'tool' in decs:
                    return node.name
    except SyntaxError:
        pass
    # Fallback: regex
    m = re.search(r"def\s+(\w+)\s*\(", code)
    return m.group(1) if m else None


def _validate_tool_code(code: str, name: str) -> tuple[bool, str]:
    """Check generated tool code is real (not a stub). Returns (valid, reason)."""
    stub_patterns = [
        (r"^\s*pass\s*$",                              "bare `pass` at top level (not inside try)"),
        (r"^\s*\.\.\.\s*$",                            "bare `...`"),
        (r"raise NotImplementedError",                 "NotImplementedError"),
        (r"#\s*(TODO|FIXME|implement me)",             "TODO/placeholder comment"),
        (r"return f?['\"].*stub.*['\"]",               "stub return string"),
        (r"return f?['\"].*not implemented",           "not-implemented return"),
        (r"from duckduckgo_search import ddg\b",       "uses deprecated ddg() function"),
        (r"(?i)#.*simulate",                           "simulated logic comment"),
        (r"is_deepfake\s*=\s*(True|False)",            "hardcoded boolean logic"),
    ]
    # Check stub patterns — but NOT bare `pass` inside try/except blocks (valid in real+mock pattern)
    for pattern, reason in stub_patterns:
        if pattern == r"^\s*pass\s*$":
            # Allow `pass` only inside except blocks (the real+mock pattern uses it)
            # Flag it only if the entire function body is just pass
            try:
                tree = ast.parse(code)
                for node in ast.walk(tree):
                    if isinstance(node, ast.FunctionDef) and node.name == name:
                        non_pass = [s for s in node.body if not isinstance(s, ast.Pass)
                                    and not (isinstance(s, ast.Expr) and isinstance(s.value, ast.Constant))]
                        if not non_pass:
                            return False, "function body is only pass/docstring"
            except SyntaxError:
                pass
        else:
            if re.search(pattern, code, re.MULTILINE | re.IGNORECASE):
                return False, reason
    try:
        tree = ast.parse(code)
        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef) and node.name == name:
                real_stmts = [
                    s for s in node.body
                    if not (isinstance(s, ast.Expr) and isinstance(s.value, ast.Constant))
                    and not isinstance(s, ast.Pass)
                ]
                if not real_stmts:
                    return False, "empty function body"
    except SyntaxError as e:
        return False, f"syntax error: {e}"
    return True, "ok"


def _regenerate_single_tool(name: str, purpose: str, task: str) -> str:
    """Regenerate one tool with real+mock pattern. Falls back to DuckDuckGo real+mock."""
    prompt = (
        f"Write a single FULLY IMPLEMENTED Python @tool function with REAL+MOCK pattern.\n"
        f"Tool name : {name}\n"
        f"  The def line MUST be: def {name}(...)\n"
        f"Purpose   : {purpose}\n"
        f"Task      : {task}\n\n"
        "Rules:\n"
        "- @tool decorator from langchain_core.tools\n"
        "- Docstring immediately after def line\n"
        "- ALL imports inside function body\n"
        "- Return str always\n"
        "- REAL+MOCK pattern: try real implementation first, fall back to mock data on failure\n"
        "- No raise NotImplementedError\n"
        "- requests/beautifulsoup4/duckduckgo-search/stdlib only\n"
        "- DuckDuckGo: `from duckduckgo_search import DDGS` + `with DDGS() as ddgs: ddgs.text(...)`\n"
        "  NEVER use `from duckduckgo_search import ddg`\n\n"
        "Respond with ONLY the Python function source — no JSON, no markdown."
    )
    llm_hot = ChatOpenAI(openai_api_key=OPENAI_API_KEY, model="gpt-4o", temperature=0.5)
    try:
        resp = invoke_with_backoff(llm_hot, [HumanMessage(content=prompt)])
        code = re.sub(r"```(?:python)?", "", resp.content or "").replace("```", "").strip()
        actual_name = _extract_actual_fn_name(code)
        if actual_name and actual_name != name:
            code = re.sub(
                rf"\bdef\s+{re.escape(actual_name)}\s*\(",
                f"def {name}(",
                code, count=1
            )
        return code
    except Exception:
        # Real+mock DuckDuckGo fallback
        return (
            f"@tool\n"
            f"def {name}(query: str) -> str:\n"
            f'    """{purpose}"""\n'
            f"    # ── REAL implementation ──────────────────────────────────\n"
            f"    try:\n"
            f"        from duckduckgo_search import DDGS\n"
            f"        with DDGS() as ddgs:\n"
            f"            results = list(ddgs.text(query, max_results=4))\n"
            f"        if results:\n"
            f"            return '\\n'.join(\n"
            f"                f\"{{r['title']}}: {{r['body']}}\" for r in results\n"
            f"            )\n"
            f"    except Exception:\n"
            f"        pass  # fall through to mock\n"
            f"    # ── MOCK fallback ─────────────────────────────────────────\n"
            f"    return (\n"
            f"        f\"[MOCK] Results for '{{query}}':\\n\"\n"
            f"        f\"1. Overview of {{query}} — key concepts.\\n\"\n"
            f"        f\"2. {{query}} latest developments.\\n\"\n"
            f"        f\"3. How {{query}} works in practice.\"\n"
            f"    )\n"
        )


def generate_tools_for_task(task: str, tool_specs: list[dict]) -> list[dict]:
    """
    Given [{name, purpose}] specs, generate fully-implemented @tool functions.
    Validates each; retries stubs. Returns [{name, description, packages, code, actual_fn_name}].
    """
    # Sanitize: remap package names to semantic tool names
    tool_specs = _sanitize_tool_specs(tool_specs)

    specs_text = "\n".join(
        f"  {i+1}. name={s['name']}, purpose={s['purpose']}"
        for i, s in enumerate(tool_specs)
    )
    prompt = (
        f"Task: {task}\n\n"
        f"Generate these {len(tool_specs)} tools:\n{specs_text}\n\n"
        "IMPORTANT: Each generated function's def line MUST match its name field exactly.\n"
        "Return a JSON array with one object per tool. No markdown."
    )

    def _attempt(temperature: float) -> list | None:
        _llm = ChatOpenAI(openai_api_key=OPENAI_API_KEY, model="gpt-4o", temperature=temperature)
        try:
            resp    = invoke_with_backoff(_llm, [
                SystemMessage(content=TOOL_GEN_SYSTEM_PROMPT),
                HumanMessage(content=prompt)
            ])
            cleaned = re.sub(r"```(?:json)?", "", resp.content or "").replace("```", "").strip()
            tools   = json.loads(cleaned)
            assert isinstance(tools, list)
            return tools
        except Exception:
            return None

    tools = _attempt(0.2) or _attempt(0.7) or []

    # Validate each; regenerate stubs individually
    validated = []
    for t in tools:
        name = t.get("name", "unknown")
        code = t.get("code", "")
        # Check function name matches JSON name; fix if not
        actual_fn = _extract_actual_fn_name(code)
        if actual_fn and actual_fn != name:
            print(f"  [ToolGen] name mismatch: JSON says '{name}', def says '{actual_fn}' — renaming def")
            code = re.sub(
                rf"\bdef\s+{re.escape(actual_fn)}\s*\(",
                f"def {name}(",
                code, count=1
            )
            t["code"] = code

        valid, reason = _validate_tool_code(code, name)
        if not valid:
            print(f"  [ToolGen] stub detected in '{name}' ({reason}), regenerating…")
            t["code"] = _regenerate_single_tool(name, t.get("description", ""), task)
        validated.append(t)

    # Add any tools missing from output
    generated_names = {t.get("name") for t in validated}
    for spec in tool_specs:
        if spec["name"] not in generated_names:
            print(f"  [ToolGen] '{spec['name']}' missing, generating standalone…")
            validated.append({
                "name":        spec["name"],
                "description": spec["purpose"],
                "packages":    [],
                "code":        _regenerate_single_tool(spec["name"], spec["purpose"], task),
            })

    return validated


def build_tools_py(tool_defs: list[dict]) -> tuple[str, list[str]]:
    """
    Assemble tools.py source from generated tool dicts.
    Returns (tools_py_source, list_of_actual_function_names).
    The actual function names are extracted from the AST — not the spec names —
    so ALL_TOOLS and agent.py imports always reference real function names.
    """
    header = textwrap.dedent("""\
        # tools.py — Generated by AgentForge v3
        # Real, task-specific implementations. No mocks, no stubs.
        from langchain_core.tools import tool

    """)
    parts = []
    actual_names = []

    for t in tool_defs:
        code = t.get("code", "").strip()
        if not code.startswith("@tool"):
            code = "@tool\n" + code

        # Extract the ACTUAL function name from AST (ground truth)
        actual_fn = _extract_actual_fn_name(code)
        if not actual_fn:
            actual_fn = t.get("name", "unknown_tool")

        actual_names.append(actual_fn)
        parts.append(f"# ── {actual_fn} ──\n{code}\n")

    footer = f"\nALL_TOOLS = [{', '.join(actual_names)}]\n"
    source = header + "\n".join(parts) + footer
    return source, actual_names

Overwriting tool_gen.py
